Chargement des données d'entrainement

In [4]:
import pandas as pd
import numpy as np

df_work = pd.read_csv('../data/segment_alerts_all_airports_train.csv')

Paramètre utile 

In [5]:
ZONE=20 #zone de sécurité où l'alerte se déclache
RISQUE_KM=20 # la distance pour laquelle les éclair sont jugé comme dangereux dans le calcul du risque
# Risque = éclair à moins de N km raté / éclair à moins de N km totale
RISQUE_POURCENT=0.02 # Le pourcentage de risque qu'on est prêt a toléré à l'entrainement

Traitement des données et création des features 

In [6]:
FEATURES = [
    # 📍 1. Spatiales & Localisation
    'lon', 'lat', 'maxis', 'dist', 'azimuth', 

    # ⏱️ 2. Temporelles (Absolues et Cycliques)
    'time_since_last_strike', 'time_since_storm_start',
    'time_since_last_N', 
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos',

    # 🏎️ 3. Cinématiques (Mouvement de l'orage)
    'delta_dist', 'delta_azimuth', 'delta_azimuth_norm', 'speed', 'acceleration',

    # ⚡ 4. Intensité & Type
    'abs_amplitude', 'polarity',

    # 📈 5. Ratios, Comptages & Activité immédiate
    'strikes_last_5min', 'strikes_last_10min', 'strikes_last_20min', 'activity_trend',
    'cum_n_strikes', 'cum_cg_strikes', 'ratio_n_r_cumul',
    'cumulative_count', 

    # 📚 6. Statistiques Cumulées (Historique de l'orage jusqu'à l'instant T)
    'mean_dist_so_far', 'std_dist_so_far', 'mean_amp_so_far',

    # 🔄 7. Tendances à Court Terme (Rolling Features)
    'rolling_icloud_mean', 'rolling_amp_mean', 'rolling_delta_dist_sum',
    'rolling_time_diff_mean', 'rolling_min_dist', 'rolling_dist_std',
    'rolling_azimuth_std', 

    # 🌍 8. Comparaisons Globales (Storm-level vs instant T)
    'amp_vs_storm_mean', 'amp_vs_storm_max',
    'storm_max_amplitude', 

    # 🎯 --- CATÉGORIQUE / EMBEDDING (OBLIGATOIREMENT À LA TOUTE FIN) ---
    'airport_code'
]

import pandas as pd
import numpy as np


# =========================================================
# CRÉATION DES GROUPES D'ORAGES (SÉQUENCES ET CONTEXTE) - VERSION OFFICIELLE
# =========================================================
def add_storm_groups_by_target(df, context_size=5, time_window_minutes=30):
    """
    Crée les groupes d'orages en se basant STRICTEMENT sur la colonne de cible officielle.
    Modifie le DataFrame `df` EN PLACE.
    """
    # Sécurité temporelle
    if not pd.api.types.is_datetime64_any_dtype(df['date']):
        df['date'] = pd.to_datetime(df['date'])

    # Tri chronologique et réinitialisation de l'index EN PLACE
    df.sort_values(['airport', 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Initialisation à -1 (Bruit par défaut)
    df['storm_group_id'] = -1

    # On repère où sont les cibles (les points de repère de l'alerte)
    is_target = df['is_last_lightning_cloud_ground'].notna()

    if not is_target.any():
        print("⚠️ Aucun groupe valide trouvé. Toutes les lignes sont à -1.")
        return

    is_true = df['is_last_lightning_cloud_ground'].isin([True, 1, 1.0, 'True'])

    # Marquer le début des séquences "brutes"
    airport_changed = df['airport'] != df['airport'].shift(1)
    after_true = is_true.shift(1).fillna(False)
    starts_new_group = airport_changed | after_true

    # On crée un ID brut pour toute la séquence (y compris le bruit du début)
    raw_group_id = starts_new_group.cumsum()

    valid_indices = []

    # On parcourt chaque séquence brute
    for grp_id, group_df in df.groupby(raw_group_id):
        targets = group_df['is_last_lightning_cloud_ground'].notna()

        if not targets.any():
            continue  # Pas de cible dans cette séquence brute

        # Index et heure du tout premier déclenchement de l'alerte
        first_target_idx = targets.idxmax()
        first_target_time = group_df.loc[first_target_idx, 'date']

        # 🚀 CORRECTION DU BUG DE L'INDEX -1 :
        if first_target_idx == group_df.index[0]:
            before_df = pd.DataFrame(columns=group_df.columns)
        else:
            before_df = group_df.loc[:first_target_idx - 1]

        # Filtre temporel : Uniquement dans la fenêtre demandée (ex: 30 min)
        if not before_df.empty:
            time_diffs = first_target_time - before_df['date']
            valid_context = before_df[time_diffs <= pd.Timedelta(minutes=time_window_minutes)]
            context_indices = valid_context.index[-context_size:].tolist()
        else:
            context_indices = []

        # Les index de l'alerte officielle (jusqu'à la fin du groupe)
        alert_indices = group_df.loc[first_target_idx:].index.tolist()

        # On rassemble les index validés
        valid_indices.extend(context_indices + alert_indices)

    # On assigne les vrais IDs uniquement aux lignes validées
    df.loc[valid_indices, 'storm_group_id'] = raw_group_id[valid_indices]

    # Renumérotation propre
    valid_mask = df['storm_group_id'] != -1
    if valid_mask.any():
        df.loc[valid_mask, 'storm_group_id'] = pd.factorize(df.loc[valid_mask, 'storm_group_id'])[0] + 1

    print("✅ Groupes d'orages restaurés selon la cible officielle du Dataset !")


# =========================================================
# CALCUL DE LA CIBLE (TIME TO END)
# =========================================================
def add_time_to_end_target(df):
    """
    Calcule le temps restant (en minutes) avant la levée de l'alerte.
    Note Anti-Leakage : Il est normal et OBLIGATOIRE que cette fonction regarde
    dans le futur (bfill), car elle calcule la Target (Y) que l'IA devra deviner.
    """
    print("⏳ Calcul de la cible de Régression (RUL) par aéroport...")

    df['date'] = pd.to_datetime(df['date'])
    df.sort_values(by=['airport', 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Identifier l'heure exacte des éclairs qui marquent la fin
    df['next_true_time'] = df['date'].where(df['is_last_lightning_cloud_ground'].isin([True, 1, 1.0, 'True']))

    # Remplir vers le haut (bfill) EN GROUPANT PAR ORAGE
    df['next_true_time'] = df.groupby('storm_group_id')['next_true_time'].bfill()

    # Calcul de l'écart temporel en minutes + 30 minutes réglementaires
    df['time_to_end'] = (df['next_true_time'] - df['date']).dt.total_seconds() / 60.0 + 30.0

    # 🚀 CORRECTION : Suppression de la ligne qui mettait des NaN sur les éclairs intra-nuageux.
    # Un réseau de neurones crashe si sa Target contient des NaN !

    # Nettoyage
    df.drop(columns=['next_true_time'], inplace=True)

    if 'storm_group_id' in df.columns:
        df.sort_values(by=['storm_group_id', 'date'], inplace=True)
        df.reset_index(drop=True, inplace=True)


# =========================================================
# FILTRAGE DES ORAGES UTILES
# =========================================================
def filter_useful_storms(df):
    """
    Garde uniquement les orages qui ont au moins un éclair au sol (CG)
    à moins de 20 km (les orages qui justifient vraiment une alerte).
    """
    initial_rows = len(df)
    initial_groups = df['storm_group_id'].nunique()

    # Identification des groupes valides
    is_alert_strike = (df['icloud'] == 0) & (df['dist'] <= ZONE)
    valid_group_ids = df.loc[is_alert_strike, 'storm_group_id'].unique()

    # Filtrage IN-PLACE
    mask_to_keep = df['storm_group_id'].isin(valid_group_ids)
    final_rows = mask_to_keep.sum()
    final_groups = len(valid_group_ids)

    df.drop(df[~mask_to_keep].index, inplace=True)

    # Réarrangement des IDs
    df['storm_group_id'] = pd.factorize(df['storm_group_id'])[0] + 1
    df.reset_index(drop=True, inplace=True)

    deleted_rows = initial_rows - final_rows
    deleted_groups = initial_groups - final_groups

    print("-" * 40)
    print("⚡ RAPPORT DE FILTRAGE DES ORAGES ⚡")
    print("-" * 40)
    print(f"Groupes : {initial_groups} -> {final_groups} (-{deleted_groups})")
    print(f"Lignes  : {initial_rows} -> {final_rows} (-{deleted_rows})")
    print("-" * 40)


# =========================================================
# NETTOYAGE DES ANOMALIES CAPTEURS
# =========================================================
def remove_pise_2016(df):
    """
    Supprime les données de l'aéroport de Pise pour l'année 2016
    (anomalie capteur Météorage).
    """
    if not pd.api.types.is_datetime64_any_dtype(df['date']):
        df['date'] = pd.to_datetime(df['date'])

    mask_to_drop = (df['airport'] == 'Pise') & (df['date'].dt.year == 2016)
    lignes_a_supprimer = mask_to_drop.sum()

    if lignes_a_supprimer > 0:
        df.drop(df[mask_to_drop].index, inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f"🧹 NETTOYAGE : Suppression de {lignes_a_supprimer} lignes pour Pise (2016).")
    else:
        print("✅ Aucune donnée de Pise 2016 n'a été trouvée/supprimée.")


# =========================================================
# SUPPRESSION DU BRUIT (HORS SÉQUENCES)
# =========================================================
def remove_noise_data(df):
    """
    Supprime toutes les lignes qui n'appartiennent pas à un groupe valide (-1).
    """
    initial_rows = len(df)
    mask_noise = df['storm_group_id'] == -1
    rows_to_drop = mask_noise.sum()

    if rows_to_drop > 0:
        df.drop(df[mask_noise].index, inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f"🧹 NETTOYAGE : Bruit supprimé ({rows_to_drop} lignes). Lignes restantes : {len(df)}")
    else:
        print("✅ Aucun bruit trouvé.")


# =========================================================
# CRÉATION DES FEATURES SPATIALES
# =========================================================
def add_zone_features(df,ZONE):
    """Indique si l'éclair a frappé dans la zone critique des 20km."""
    df['is_in_20km'] = (df['dist'] <= ZONE).astype(int)


# =========================================================
# FORMATAGE : IDENTIFIANTS D'ALERTE
# =========================================================
def format_alert_id(df):
    df['airport_alert_id'] = df['airport_alert_id'].fillna(0).astype(int)


# =========================================================
# FORMATAGE : CIBLE FIN D'ALERTE
# =========================================================
def format_last_lightning(df):
    """
    ⚠️ Ne lancer cette fonction qu'APRÈS la création des storm_groups !
    """
    df['is_last_lightning_cloud_ground'] = df['is_last_lightning_cloud_ground'].fillna(False).astype(int)


# =========================================================
# FORMATAGE : TYPE D'ÉCLAIR (INTRA-NUAGEUX / SOL)
# =========================================================
def format_icloud(df):
    df['icloud'] = df['icloud'].astype(int)


# =========================================================
# FORMATAGE : DATES
# =========================================================
def format_date(df):
    df['date'] = pd.to_datetime(df['date'])

import numpy as np
import pandas as pd

STORM_GROUP_COL = 'storm_group_id'


# =========================================================
# PRÉPARATION : TRI DES SÉQUENCES
# =========================================================
def sort_for_sequences(df):
    """Trie le dataset chronologiquement par orage."""
    df.sort_values([STORM_GROUP_COL, 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)


# =========================================================
# FEATURES TEMPORELLES
# =========================================================
def add_temporal_features(df):
    """Calcule les écarts de temps entre éclairs et depuis le début de l'orage."""
    df['time_since_last_strike'] = df.groupby(STORM_GROUP_COL)['date'].diff().dt.total_seconds().fillna(0)

    # transform('min') est safe ici car le début de l'orage est un point de repère fixe dans le passé
    min_dates = df.groupby(STORM_GROUP_COL)['date'].transform('min')
    df['time_since_storm_start'] = (df['date'] - min_dates).dt.total_seconds() / 60.0


# =========================================================
# FEATURES CINÉMATIQUES (MOUVEMENT)
# =========================================================
def add_kinematic_features(df):
    """Calcule les déplacements (distance et angle) entre éclairs consécutifs."""
    df['delta_dist'] = df.groupby(STORM_GROUP_COL)['dist'].diff().fillna(0)
    df['delta_azimuth'] = df.groupby(STORM_GROUP_COL)['azimuth'].diff().fillna(0)


# =========================================================
# FEATURES D'INTENSITÉ (AMPLITUDE)
# =========================================================
def add_intensity_features(df):
    """Extrait l'amplitude absolue et la polarité de l'éclair."""
    df['abs_amplitude'] = df['amplitude'].abs()
    df['polarity'] = np.sign(df['amplitude']).astype(int)


# =========================================================
# FEATURES CUMULÉES (HISTORIQUE DE L'ORAGE)
# =========================================================
def add_cumulative_features(df):
    """
    Calcule les statistiques cumulées depuis le début de l'orage jusqu'à l'éclair actuel.
    L'utilisation de .expanding() garantit l'absence de leakage vers le futur.
    """
    df.sort_values([STORM_GROUP_COL, 'date'], inplace=True)
    df['storm_duration_so_far'] = df['time_since_storm_start']
    df['cumulative_count'] = df.groupby(STORM_GROUP_COL).cumcount() + 1

    df['mean_dist_so_far'] = df.groupby(STORM_GROUP_COL)['dist'].expanding().mean().reset_index(level=0, drop=True)
    df['mean_amp_so_far'] = df.groupby(STORM_GROUP_COL)['amplitude'].expanding().mean().reset_index(level=0, drop=True)

    df['std_dist_so_far'] = df.groupby(STORM_GROUP_COL)['dist'].expanding().std().reset_index(level=0, drop=True)
    df['std_dist_so_far'] = df['std_dist_so_far'].fillna(0.0)
    print("✅ Cumulative features ajoutées (sans leakage)")


# =========================================================
# INCERTITUDE SPATIALE
# =========================================================
def add_uncertainty_features(df):
    """Estime l'aire d'incertitude de localisation de l'éclair."""
    df['error_area_km2'] = np.pi * (df['maxis'] ** 2)


# =========================================================
# ROLLING FEATURES (TENDANCES À COURT TERME)
# =========================================================
def add_rolling_features(df, length):
    """Calcule les moyennes glissantes sur les N derniers éclairs."""
    print("Calcul des Rolling Features (Tendances)...")
    df['rolling_icloud_mean'] = df.groupby(STORM_GROUP_COL)['icloud'].transform(
        lambda x: x.rolling(window=length, min_periods=1).mean()
    )
    df['rolling_amp_mean'] = df.groupby(STORM_GROUP_COL)['abs_amplitude'].transform(
        lambda x: x.rolling(window=length, min_periods=1).mean()
    )
    df['rolling_delta_dist_sum'] = df.groupby(STORM_GROUP_COL)['delta_dist'].transform(
        lambda x: x.rolling(window=length, min_periods=1).sum()
    )
    df['rolling_time_diff_mean'] = df.groupby(STORM_GROUP_COL)['time_since_last_strike'].transform(
        lambda x: x.rolling(window=length, min_periods=1).mean()
    )


# =========================================================
# ENCODAGE CYCLIQUE DU TEMPS
# =========================================================
def add_cyclical_time_features(df):
    """Transforme l'heure et la date en coordonnées circulaires (sin/cos)."""
    month = df['date'].dt.month
    hour = df['date'].dt.hour
    day_of_year = df['date'].dt.dayofyear

    df['month_sin'] = np.sin(2 * np.pi * month / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * month / 12.0)
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24.0)
    df['day_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    df['day_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)


# =========================================================
# ACTIVITÉ RÉCENTE (STRIKE RATES)
# =========================================================
def add_activity_rate_features(df):
    """Compte le nombre d'éclairs dans les X dernières minutes strictes (closed='left')."""
    df['date_for_rolling'] = df['date']

    def count_strikes_last_n_min(group, minutes):
        group = group.set_index('date_for_rolling')
        return group['icloud'].rolling(f'{minutes}min', closed='left').count().values

    for minutes in [5, 10, 20]:
        df[f'strikes_last_{minutes}min'] = df.groupby(STORM_GROUP_COL, group_keys=False).apply(
            lambda g: pd.Series(count_strikes_last_n_min(g, minutes), index=g.index)
        )

    df.drop(columns=['date_for_rolling'], inplace=True)
    df['activity_trend'] = df['strikes_last_5min'] / (df['strikes_last_10min'] + 1e-6)


# =========================================================
# STABILITÉ SPATIALE
# =========================================================
def add_spatial_stability_features(df, length=5):
    """Mesure la dispersion géographique récente de l'orage."""
    df['rolling_min_dist'] = df.groupby(STORM_GROUP_COL)['dist'].transform(
        lambda x: x.rolling(window=length, min_periods=1).min()
    )
    df['rolling_dist_std'] = df.groupby(STORM_GROUP_COL)['dist'].transform(
        lambda x: x.rolling(window=length, min_periods=1).std().fillna(0)
    )
    df['rolling_azimuth_std'] = df.groupby(STORM_GROUP_COL)['azimuth'].transform(
        lambda x: x.rolling(window=length, min_periods=1).std().fillna(0)
    )


# =========================================================
# RATIOS DE TYPES D'ÉCLAIRS (N vs CG)
# =========================================================
def add_ratio_features(df, length):
    """Calcule la proportion d'éclairs intra-nuageux par rapport aux éclairs au sol."""
    df['cum_n_strikes'] = (df['icloud'] == 1).astype(int).groupby(df[STORM_GROUP_COL]).cumsum()
    df['cum_cg_strikes'] = (df['icloud'] == 0).astype(int).groupby(df[STORM_GROUP_COL]).cumsum()

    df['ratio_n_r_cumul'] = df['cum_n_strikes'] / (df['cum_cg_strikes'] + 1e-6)
    df['rolling_n_ratio'] = df.groupby(STORM_GROUP_COL)['icloud'].transform(
        lambda x: x.rolling(window=length, min_periods=1).mean()
    )


# =========================================================
# VITESSE ET ACCÉLÉRATION DE L'ORAGE
# =========================================================
def add_velocity_features(df):
    """Estime la vitesse apparente de rapprochement/éloignement."""
    df['speed'] = df['delta_dist'] / (df['time_since_last_strike'] / 60.0 + 1e-6)
    df['speed'] = df['speed'].replace([np.inf, -np.inf], 0.0)  # Sécurité anti-crash

    df['acceleration'] = df.groupby(STORM_GROUP_COL)['speed'].diff().fillna(0)
    df['delta_azimuth_norm'] = (df['delta_azimuth'] + 180) % 360 - 180


# =========================================================
# AMPLITUDE RELATIVE AU PASSÉ
# =========================================================
def add_relative_amplitude_features(df):
    """Compare l'amplitude de l'éclair actuel à la moyenne/max historique de l'orage."""
    df.sort_values([STORM_GROUP_COL, 'date'], inplace=True)

    df['expanding_amp_mean'] = df.groupby(STORM_GROUP_COL)['amplitude'].expanding().mean().reset_index(level=0,
                                                                                                       drop=True)
    df['expanding_amp_max'] = df.groupby(STORM_GROUP_COL)['abs_amplitude'].expanding().max().reset_index(level=0,drop=True)

    df['amp_vs_storm_mean'] = df['amplitude'] / df['expanding_amp_mean'].replace(0, np.nan)
    df['amp_vs_storm_max'] = df['abs_amplitude'] / df['expanding_amp_max'].replace(0, np.nan)

    df.drop(columns=['expanding_amp_mean', 'expanding_amp_max'], inplace=True)
    df['amp_vs_storm_mean'] = df['amp_vs_storm_mean'].fillna(1.0)
    df['amp_vs_storm_max'] = df['amp_vs_storm_max'].fillna(1.0)
    print("✅ Relative amplitude features ajoutées (sans leakage)")


# =========================================================
# POSITION TEMPORELLE DE L'ÉCLAIR
# =========================================================
def add_position_features(df):
    """Position ordinale dans la séquence et temps depuis le dernier intra-nuageux (N)."""
    df['position_in_storm'] = df.groupby(STORM_GROUP_COL).cumcount()

    # Marquer la date des N uniquement
    is_N = (df['icloud'] == 1)
    df['_last_N_date'] = df['date'].where(is_N)

    # Forward fill PAR ORAGE — propage à tous les éclairs suivants (R et N) sans fuite
    df['_last_N_date'] = df.groupby(STORM_GROUP_COL)['_last_N_date'].ffill()

    # Calcul du temps écoulé depuis ce dernier N
    df['time_since_last_N'] = (df['date'] - df['_last_N_date']).dt.total_seconds() / 60.0

    # -1 = pas encore de N dans cet orage
    df['time_since_last_N'] = df['time_since_last_N'].fillna(-1)

    df.drop(columns=['_last_N_date'], inplace=True)
    print("✅ time_since_last_N ajouté (propagé à tous les éclairs)")


# =========================================================
# SURVIE ET DANGEROSITÉ IMMÉDIATE (DERNIER R)
# =========================================================
def add_survival_features(df):
    """
    Temps depuis le dernier éclair sol (R) dans la zone d'alerte (<20km).
    Essentiel pour estimer si l'orage est en train de se dissiper.
    """
    # Marquer la date des R uniquement
    is_R = (df['icloud'] == 0) & (df['dist'] <= ZONE)
    df['_last_R_date'] = df['date'].where(is_R)

    # Forward fill PAR ORAGE
    df['_last_R_date'] = df.groupby(STORM_GROUP_COL)['_last_R_date'].ffill()

    # Calcul du temps écoulé depuis ce dernier R
    df['time_since_last_R'] = (df['date'] - df['_last_R_date']).dt.total_seconds() / 60.0

    # -1 = pas encore de R dans cet orage (contexte N avant le premier R)
    df['time_since_last_R'] = df['time_since_last_R'].fillna(-1)

    df.drop(columns=['_last_R_date'], inplace=True)
    print("✅ time_since_last_R ajouté (propagé à tous les éclairs)")


# =========================================================
# STATISTIQUES GLOBALES DE L'ORAGE (SO FAR)
# =========================================================
def add_storm_level_features(df):
    """Caractéristiques globales de l'orage jusqu'à l'instant T (sans lire le futur)."""
    df.sort_values([STORM_GROUP_COL, 'date'], inplace=True)
    df['storm_total_lightnings'] = df.groupby(STORM_GROUP_COL).cumcount() + 1

    df['_icloud_cumsum'] = df.groupby(STORM_GROUP_COL)['icloud'].cumsum()
    df['storm_cloud_ratio'] = df['_icloud_cumsum'] / df['storm_total_lightnings']
    df.drop(columns=['_icloud_cumsum'], inplace=True)

    df['storm_mean_amplitude'] = df.groupby(STORM_GROUP_COL)['amplitude'].expanding().mean().reset_index(level=0,
                                                                                                         drop=True)
    df['storm_max_amplitude'] = df.groupby(STORM_GROUP_COL)['abs_amplitude'].expanding().max().reset_index(level=0,
                                                                                                           drop=True)
    df['storm_mean_dist'] = df.groupby(STORM_GROUP_COL)['dist'].expanding().mean().reset_index(level=0, drop=True)

    print("✅ Storm-level features ajoutées (sans leakage)")


# =========================================================
# EMBEDDING AÉROPORT
# =========================================================
def add_airport_code(df):
    """Encode le nom de l'aéroport en entier pour l'Embedding PyTorch."""
    mapping = {'Bron': 0, 'Bastia': 1, 'Ajaccio': 2, 'Nantes': 3, 'Pise': 4, 'Biarritz': 5}
    df['airport_code'] = df['airport'].map(mapping)
    print("✅ Codes aéroports ajoutés pour l'Embedding")


# =========================================================
# NETTOYAGE FINAL DES NAN
# =========================================================
def fill_nan_features(df):
    """Remplace les valeurs manquantes générées par les calculs glissants/cumulés."""
    fills = {
        'strikes_last_5min': 0,
        'strikes_last_10min': 0,
        'strikes_last_20min': 0,
        'activity_trend': 1.0,
        'rolling_min_dist': df['dist'] if 'dist' in df.columns else 0,  # Utilise la dist actuelle si pas d'historique
        'rolling_dist_std': 0,
        'rolling_azimuth_std': 0,
        'cum_n_strikes': 0,
        'cum_cg_strikes': 0,
        'ratio_n_r_cumul': 0,
        'rolling_n_ratio': 0,
        'speed': 0,
        'acceleration': 0,
        'delta_azimuth_norm': 0,
        'amp_vs_storm_mean': 1.0,
        'amp_vs_storm_max': 1.0,
        'position_in_storm': 0,
        'time_since_last_N': -1,
        'time_since_last_R': -1,
        'storm_total_lightnings': 1,
        'storm_cloud_ratio': 0.5,
        'storm_mean_amplitude': 0,
        'storm_max_amplitude': 0,
        'storm_mean_dist': 0,
        'cumulative_count': 1,
        'mean_dist_so_far': 0,
        'mean_amp_so_far': 0,
        'std_dist_so_far': 0,
    }

    for col, val in fills.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)

    valid_cols = [c for c in fills.keys() if c in df.columns]
    print(f"✅ NaN nettoyés. Vérification : {df[valid_cols].isna().sum().sum()} NaN restants dans les features.")



# =========================================================
# VÉRIFICATION DE SÉCURITÉ ANTI-LEAKAGE
# =========================================================
def check_no_future_leakage(df, group_col=STORM_GROUP_COL):
    """
    Vérifie qu'aucune feature calculée n'est 'magiquement' corrélée
    à la cible du futur de façon suspecte.
    """
    print("🔍 Vérification anti-leakage...")
    suspicious = []

    for col in df.select_dtypes(include=[np.number]).columns:
        # 🚀 CORRECTION : On ignore la colonne de groupe pour éviter le KeyError !
        if col in ['time_to_end', 'lightning_id', 'lightning_airport_id', group_col]:
            continue

        first_values = df.groupby(group_col).first()[col]
        if first_values.isna().all():
            continue

        if 'time_to_end' in df.columns:
            corr = df[col].corr(df['time_to_end'])
            # Une corrélation au-delà de 0.85 sur une métrique temporelle est suspecte
            if abs(corr) > 0.85:
                suspicious.append((col, corr))

    if suspicious:
        print("⚠️ Features suspectes (corrélation > 0.85 avec target):")
        for col, corr in sorted(suspicious, key=lambda x: -abs(x[1])):
            print(f"   {col}: corr = {corr:.3f}")
    else:
        print("✅ Aucune feature suspecte détectée. Le dataset est safe.")

format_date(df_work)
add_storm_groups_by_target(df_work, context_size=20, time_window_minutes=60)
add_time_to_end_target(df_work)
add_zone_features(df_work, ZONE)
format_alert_id(df_work)
format_last_lightning(df_work)
format_icloud(df_work)

remove_pise_2016(df_work)
remove_noise_data(df_work)
filter_useful_storms(df_work)


sort_for_sequences(df_work)
add_temporal_features(df_work)
add_kinematic_features(df_work)
add_intensity_features(df_work)
add_cumulative_features(df_work)
add_uncertainty_features(df_work)

rolling_window = 10
add_rolling_features(df_work, length=rolling_window)

add_cyclical_time_features(df_work)
add_activity_rate_features(df_work)
add_spatial_stability_features(df_work, length=rolling_window)
add_ratio_features(df_work, length=rolling_window)
add_velocity_features(df_work)
add_relative_amplitude_features(df_work)

add_position_features(df_work)
add_survival_features(df_work)
add_storm_level_features(df_work)
add_airport_code(df_work)


fill_nan_features(df_work)
check_no_future_leakage(df_work)
from sklearn.preprocessing import LabelEncoder

# 1. Créer l'encodeur
le = LabelEncoder()

# 2. Transformer la colonne 'airport' en entiers
# On crée une nouvelle colonne 'airport_id' pour garder l'originale intacte
df_work['airport_id'] = le.fit_transform(df_work['airport'])

# 3. Afficher la correspondance (pour savoir quel chiffre correspond à quel aéroport)
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Correspondance des aéroports :")
print(mapping)
# Convertir l'azimuth en radians (si ton azimuth est en degrés)
df_work['azimuth_rad'] = np.radians(df_work['azimuth'])

# Calculer X et Y (Aéroport = 0,0)
df_work['X'] = df_work['dist'] * np.sin(df_work['azimuth_rad'])
df_work['Y'] = df_work['dist'] * np.cos(df_work['azimuth_rad'])

✅ Groupes d'orages restaurés selon la cible officielle du Dataset !
⏳ Calcul de la cible de Régression (RUL) par aéroport...
🧹 NETTOYAGE : Suppression de 24421 lignes pour Pise (2016).
🧹 NETTOYAGE : Bruit supprimé (64308 lignes). Lignes restantes : 418342
----------------------------------------
⚡ RAPPORT DE FILTRAGE DES ORAGES ⚡
----------------------------------------
Groupes : 2400 -> 2400 (-0)
Lignes  : 418342 -> 418342 (-0)
----------------------------------------
✅ Cumulative features ajoutées (sans leakage)
Calcul des Rolling Features (Tendances)...
✅ Relative amplitude features ajoutées (sans leakage)
✅ time_since_last_N ajouté (propagé à tous les éclairs)
✅ time_since_last_R ajouté (propagé à tous les éclairs)
✅ Storm-level features ajoutées (sans leakage)
✅ Codes aéroports ajoutés pour l'Embedding
✅ NaN nettoyés. Vérification : 0 NaN restants dans les features.
🔍 Vérification anti-leakage...
✅ Aucune feature suspecte détectée. Le dataset est safe.
Correspondance des aéroports

Fonction pour agrégé les features du dataset minute par minutes avec création de nouvelle features

In [7]:
import pandas as pd
import numpy as np

def creer_grille_temporelle(df_raw, lookback_min=5, horizon_min=30, zone_critique=20.0, audit_dist_km=3.0):
    """
    Transforme un historique d'éclairs bruts en une grille d'évaluation minute par minute.
    Intègre un chronomètre de silence et une vision géométrique multi-échelles.
    """
    print(f"Création de la grille temporelle (Lookback: {lookback_min}m | Horizon: {horizon_min}m)...")
    
    df = df_raw.copy()
    df['date'] = pd.to_datetime(df['date'])
    
    orages_resampled = []
    
    # On groupe par aéroport et par orage pour ne pas mélanger les événements
    groupes = df.groupby(['airport_id', 'storm_group_id'])
    
    for (airport, storm_id), df_storm in groupes:
        # On s'assure que le temps est bien indexé et trié
        df_storm = df_storm.sort_values('date').set_index('date')
        # On repère les types d'éclairs
        df_storm['is_ic'] = df_storm['icloud'].astype(int)
        df_storm['is_cg'] = (~df_storm['icloud'].astype(bool)).astype(int)
        
        # ⚠️ REMPLACE 'amplitude' PAR LE VRAI NOM DE TA COLONNE D'AMPLITUDE BRUTE SI BESOIN
        df_storm['is_positive'] = (df_storm['amplitude'] > 0).astype(int)

        df_storm['is_cg_audit'] = ((df_storm['is_cg'] == 1) & (df_storm['dist'] <= audit_dist_km)).astype(int)
        df_storm['is_cg_zonekm'] = ((df_storm['is_cg'] == 1) & (df_storm['dist'] <= zone_critique)).astype(int)
        mask_danger_reel = (df_storm['is_cg'] == 1) & (df_storm['dist'] <= zone_critique)
        df_storm['is_cg_3km'] = ((df_storm['is_cg'] == 1) & (df_storm['dist'] <= 3.0)).astype(int)
        if mask_danger_reel.any():
            # 2. On prend l'heure exacte du TOUT DERNIER éclair dangereux
            heure_dernier_danger = df_storm[mask_danger_reel].index.max()
            
            # 3. La règle humaine : on ferme l'aéroport pendant les 30 minutes qui suivent
            fin_alerte_humaine = heure_dernier_danger + pd.Timedelta(minutes=horizon_min)
        else:
            # Sécurité : Si l'orage n'a jamais touché le sol à <20km, l'alerte n'est pas censée exister
            fin_alerte_humaine = df_storm.index.max()
        # ==========================================
        # 🧱 BLOC 1 : AGRÉGATION À LA MINUTE & CHRONOMÈTRE
        # ==========================================
        agregations = {
            'dist': 'min',                      
            'delta_dist': 'mean',               
            'abs_amplitude': 'max',             
            'cum_n_strikes': 'max',  
            'X' : 'mean',
            'Y' : 'mean'
        }
        
        df_minute = df_storm.resample('1min').agg(agregations)
        df_minute['activity_count'] = df_storm.resample('1min').size()

        df_minute['count_cg_audit'] = df_storm['is_cg_audit'].resample('1min').sum()
        df_minute['count_cg_zonekm'] = df_storm['is_cg_zonekm'].resample('1min').sum() # NOUVEAU
        df_minute['count_cg_3km'] = df_storm['is_cg_3km'].resample('1min').sum()


        # 🎯 3. LE PADDING : On force la grille à aller jusqu'à la fin de l'alerte humaine !
        index_debut = df_minute.index.min()
        # La fin est soit la fin des capteurs, soit la fin de l'alerte (le plus tard des deux)
        index_fin = fin_alerte_humaine 
        
        nouvel_index = pd.date_range(start=index_debut, end=index_fin, freq='1min')
        df_minute = df_minute.reindex(nouvel_index)

        df_minute['count_cg_audit'] = df_minute['count_cg_audit'].fillna(0)
        df_minute['count_cg_zonekm'] = df_minute['count_cg_zonekm'].fillna(0)
        df_minute['count_cg_3km'] = df_minute['count_cg_3km'].fillna(0)

        # ⏱️ LE CHRONOMÈTRE DE SILENCE 
        df_minute['a_frappe'] = df_minute['activity_count'] > 0
        df_minute['heure_dernier_eclair'] = df_minute.index.to_series().where(df_minute['a_frappe']).ffill()
        df_minute['minutes_since_last_strike'] = (df_minute.index - df_minute['heure_dernier_eclair']).dt.total_seconds() / 60.0
        df_minute['minutes_since_last_strike'] = df_minute['minutes_since_last_strike'].fillna(0)
        
        df_minute['count_ic'] = df_storm['is_ic'].resample('1min').sum().reindex(nouvel_index, fill_value=0)
        df_minute['count_cg'] = df_storm['is_cg'].resample('1min').sum().reindex(nouvel_index, fill_value=0)
        df_minute['count_pos'] = df_storm['is_positive'].resample('1min').sum().reindex(nouvel_index, fill_value=0)
        
        df_minute['dist_max'] = df_storm['dist'].resample('1min').max().reindex(nouvel_index) # La distance max reste NaN si pas d'éclair
        
        df_minute['azimuth_std'] = df_storm['azimuth'].resample('1min').std().fillna(0).reindex(nouvel_index, fill_value=0)
        df_minute['energie_totale'] = df_storm['abs_amplitude'].resample('1min').sum().reindex(nouvel_index, fill_value=0)
        
        # ==========================================
        # 🛡️ GESTION DE LA MÉMOIRE (L'astuce anti-zombie)
        # ==========================================
        # 1. On sauvegarde la dernière distance connue dans une variable à part pour la mémoire de l'IA
        df_minute['last_known_dist'] = df_minute['dist'].ffill()
        
        # 2. On ne propage QUE les cumuls. 
        # dist, delta_dist, X, Y et dist_max restent "NaN" (vide) s'il n'y a pas d'éclair !
        colonnes_a_propager = ['cum_n_strikes']
        df_minute[colonnes_a_propager] = df_minute[colonnes_a_propager].ffill()
        
        # 3. Remplissage ciblé des zéros (On ne touche SURTOUT PAS aux distances NaN)
        colonnes_a_zero = ['abs_amplitude', 'activity_count']
        df_minute[colonnes_a_zero] = df_minute[colonnes_a_zero].fillna(0)
        
        # ==========================================
        # ⏪ BLOC 2 : LE RÉTROVISEUR MULTI-ÉCHELLES
        # ==========================================
        df_features = df_minute[['minutes_since_last_strike', 'cum_n_strikes', 'last_known_dist','count_cg_audit','count_cg_zonekm','count_cg_3km']].copy()
        
        # Court Terme (5 min)
        df_5m = df_minute.rolling(window=5, min_periods=1).agg({
            'activity_count': 'sum',             
            'dist': 'min',                       
            'delta_dist': 'mean',                
            'abs_amplitude': 'max',
            'X' : 'mean',
            'Y' : 'mean'           
        })
        df_5m.columns = [f"{col}_last_5m" for col in df_5m.columns]
        
        # Moyen Terme (20 min)
        df_20m = df_minute.rolling(window=20, min_periods=1).agg({
            'activity_count': 'sum',             
            'dist': 'min'                        
        })
        df_20m.columns = [f"{col}_last_20m" for col in df_20m.columns]
        
        # On fusionne le tout
        df_features = pd.concat([df_features, df_5m, df_20m], axis=1)
        
        # Contexte Temporel
        df_features['hour_sin'] = np.sin(2 * np.pi * df_features.index.hour / 24.0)
        df_features['hour_cos'] = np.cos(2 * np.pi * df_features.index.hour / 24.0)
        df_features['month_sin'] = np.sin(2 * np.pi * df_features.index.month / 12.0)
        df_features['month_cos'] = np.cos(2 * np.pi * df_features.index.month / 12.0)

        # ==========================================
        # 🧠 BLOC 3 : VARIABLES EXPERTES (LA GÉOMÉTRIE LISSÉE)
        # ==========================================
        # 🛡️ FIX : On empêche l'amnésie de distance pendant le padding
        df_features['dist_last_5m'] = df_features['dist_last_5m'].fillna(df_minute['last_known_dist'])
        df_features['dist_last_20m'] = df_features['dist_last_20m'].fillna(df_minute['last_known_dist'])
        # 1. Le Vrai Delta Macro : L'orage s'éloigne-t-il ? (>0 = éloignement)
        df_features['distance_macro_trend'] = df_features['dist_last_5m'] - df_features['dist_last_20m']
        
        # 2. Le Choc d'Activité : L'orage s'effondre-t-il d'un coup ?
        df_features['activity_drop_ratio'] = df_features['activity_count_last_5m'] / (df_features['activity_count_last_20m'] + 0.001)

        # Vitesse d'éloignement radiale (km par minute)
        df_features['vitesse_eloignement'] = (df_features['dist_last_5m'] - df_features['dist_last_20m']) / 15.0
        
        # Projection : Où sera l'orage dans 30 minutes s'il continue à cette vitesse ?
        df_features['dist_projetee_30m'] = df_features['dist_last_5m'] + (df_features['vitesse_eloignement'] * 30.0)

        # 1. Calcul du vecteur Vitesse (X et Y) par minute, basé sur le déplacement sur 5 min
        df_features['vitesse_X'] = (df_features['X_last_5m'] - df_features['X_last_5m'].shift(5)) / 5.0
        df_features['vitesse_Y'] = (df_features['Y_last_5m'] - df_features['Y_last_5m'].shift(5)) / 5.0
        
        # Remplir les 5 premières minutes (où shift(5) est NaN) par 0 (orage statique au début)
        df_features['vitesse_X'] = df_features['vitesse_X'].fillna(0)
        df_features['vitesse_Y'] = df_features['vitesse_Y'].fillna(0)

        # 2. Distance réelle du Centre de Gravité actuel (Hypoténuse)
        df_features['dist_centroid_actuel'] = np.sqrt(df_features['X_last_5m']**2 + df_features['Y_last_5m']**2)

        # 3. Projection Vectorielle à 30 minutes
        df_features['proj_X_30m'] = df_features['X_last_5m'] + (df_features['vitesse_X'] * 30.0)
        df_features['proj_Y_30m'] = df_features['Y_last_5m'] + (df_features['vitesse_Y'] * 30.0)
        
        # 4. Distance de la projection (À quelle distance sera le coeur de l'orage dans 30 min ?)
        df_features['dist_centroid_proj_30m'] = np.sqrt(df_features['proj_X_30m']**2 + df_features['proj_Y_30m']**2)
        
        # 5. Indice d'éloignement global (>0 = s'éloigne de l'aéroport, <0 = s'approche)
        df_features['centroid_eloignement_net'] = df_features['dist_centroid_proj_30m'] - df_features['dist_centroid_actuel']

        df_features['count_ic_last_20m'] = df_minute['count_ic'].rolling(window=20, min_periods=1).sum()
        df_features['count_cg_last_20m'] = df_minute['count_cg'].rolling(window=20, min_periods=1).sum()
        df_features['count_pos_last_20m'] = df_minute['count_pos'].rolling(window=20, min_periods=1).sum()
        
        # 🚀 LES VARIABLES MÉTÉOROLOGIQUES CLÉS
        # Ratio IC / CG (S'il s'effondre, l'orage meurt)
        df_features['ratio_ic_cg_last_20m'] = df_features['count_ic_last_20m'] / (df_features['count_cg_last_20m'] + 0.001)
        
        # Ratio de polarité positive (S'il augmente, l'enclume se vide, fin de l'orage)
        df_features['ratio_pos_last_20m'] = df_features['count_pos_last_20m'] / (df_features['activity_count_last_20m'] + 0.001)
        # 🌪️ LA TAILLE DE L'ENCLUME (Dispersion)
        # On regarde le max et la moyenne sur 5 min
        df_features['dist_max_last_5m'] = df_minute['dist_max'].rolling(window=5, min_periods=1).max()
        df_features['azimuth_std_last_5m'] = df_minute['azimuth_std'].rolling(window=5, min_periods=1).mean()
        
        # L'Étalement Radial : Différence entre l'éclair le plus loin et le plus proche (Profondeur de l'orage en km)
        df_features['storm_spread_radial'] = df_features['dist_max_last_5m'] - df_features['dist_last_5m']
        df_features['energie_last_5m'] = df_minute['energie_totale'].rolling(window=5, min_periods=1).sum()
        df_features['energie_last_20m'] = df_minute['energie_totale'].rolling(window=20, min_periods=1).sum()
        
        # Le Crash Énergétique : Si ça s'approche de 0, l'orage meurt.
        df_features['energy_drop_ratio'] = df_features['energie_last_5m'] / (df_features['energie_last_20m'] + 0.001)
        # ... (ton code actuel jusqu'à energy_drop_ratio) ...
        # 🧟‍♂️ LA PRESSION ZOMBIE (Croisement Silence x Proximité)
        # Plus l'orage est proche et silencieux, plus cet indice explose
        df_features['pression_zombie'] = df_features['minutes_since_last_strike'] / (df_features['last_known_dist'] + 1.0)

        # ==========================================
        # 🛡️ SÉCURITÉS ANTI-NaN (Pour gérer le "vide")
        # ==========================================
        # 1. Les vitesses et tendances retombent à 0 quand il n'y a pas d'éclair (l'orage est à l'arrêt)
        colonnes_vitesses_zero = [
            'distance_macro_trend', 'vitesse_eloignement', 'vitesse_X', 'vitesse_Y', 
            'centroid_eloignement_net', 'storm_spread_radial'
        ]
        df_features[colonnes_vitesses_zero] = df_features[colonnes_vitesses_zero].fillna(0)

        # 2. Les distances de projection retombent sur la "dernière distance connue" 
        # (Si on ne sait pas où il va, on suppose qu'il reste là où il était)
        df_features['dist_projetee_30m'] = df_features['dist_projetee_30m'].fillna(df_minute['last_known_dist'])
        df_features['dist_centroid_actuel'] = df_features['dist_centroid_actuel'].fillna(df_minute['last_known_dist'])
        df_features['dist_centroid_proj_30m'] = df_features['dist_centroid_proj_30m'].fillna(df_minute['last_known_dist'])
        
        # 3. ON AJOUTE LA MÉMOIRE POUR XGBOOST
        df_features['last_known_dist'] = df_minute['last_known_dist']
        # On fusionne le tout
        
        # ==========================================
        # ⏩ BLOC 4 : LE FUTUR (LA TARGET DE DANGER)
        # ==========================================
        mask_critique = (df_storm['icloud'] == False) & (df_storm['dist'] <= zone_critique)
        eclairs_critiques = df_storm[mask_critique].resample('1min').size()
        
        eclairs_critiques = eclairs_critiques.reindex(df_features.index, fill_value=0)
        
        indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=horizon_min)
        futur_critique = eclairs_critiques.rolling(window=indexer, min_periods=1).sum()
        
        df_features[f'target_danger_{horizon_min}m'] = (futur_critique > 0).astype(int)
        # ==========================================
        # 🧹 BLOC 5 : NETTOYAGE ET SÉCURITÉ SCIENTIFIQUE
        # ==========================================
        df_features['airport_id'] = airport
        df_features['storm_group_id'] = storm_id
        df_features = df_features.reset_index()
        
        
        orages_resampled.append(df_features)

    print("Assemblage final...")
    return pd.concat(orages_resampled, ignore_index=True)


Entrainement et calcul de probabilité

In [8]:
# 📋 LA LISTE PREMIUM (WHITELIST)
features_cibles = [
    # ⏱️ Chrono et Maturité
    'minutes_since_last_strike', 'cum_n_strikes',

    # ⚡ Court Terme (5 min) - Attention, 'dist_last_5m' contient maintenant des NaN !
    'activity_count_last_5m', 'dist_last_5m', 'delta_dist_last_5m', 'abs_amplitude_last_5m',

    # 🌊 Moyen Terme (20 min)
    'activity_count_last_20m', 'dist_last_20m', 'distance_macro_trend', 'activity_drop_ratio',

    # 📐 Géométrie et Projections
    'vitesse_eloignement', 'dist_projetee_30m', 'dist_centroid_actuel', 
    'dist_centroid_proj_30m', 'centroid_eloignement_net',

    # 🌩️ Physique de l'orage
    'count_ic_last_20m', 'count_cg_last_20m', 'count_pos_last_20m', 
    'ratio_ic_cg_last_20m', 'ratio_pos_last_20m',

    # 🌍 Contexte temporel
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',

    # 🔥 Les 4 Armes Nouvelles
    'azimuth_std_last_5m', 'storm_spread_radial', 'energie_last_5m', 'energy_drop_ratio',
    
    # 🧠 La Mémoire (Indispensable maintenant qu'on a enlevé le ffill)
    'last_known_dist'

]

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupKFold
import warnings

warnings.filterwarnings('ignore')

print("🚀 ÉTAPE 1 : CRÉATION DE LA GRILLE SUR TOUT LE DATASET 🚀")
# On crée la grille sur la totalité des données d'un coup (Rapide et sans Leakage !)
# Assure-toi que ton df_work est prêt, filtré et nettoyé comme d'habitude.
df_minute_all = creer_grille_temporelle(df_work, lookback_min=5, horizon_min=30, zone_critique=ZONE,audit_dist_km=RISQUE_KM)

# On enlève les colonnes potentiellement dupliquées par Jupyter
df_minute_all = df_minute_all.loc[:, ~df_minute_all.columns.duplicated()]

# On vérifie que toutes tes features cibles sont bien là
features_a_entrainer = [col for col in features_cibles if col in df_minute_all.columns]

# Définition des blocs pour le K-Fold
X_all = df_minute_all[features_a_entrainer]
y_all = df_minute_all['target_danger_30m'].squeeze()
groups_all = df_minute_all['storm_group_id']

print("\n🚀 ÉTAPE 2 : VALIDATION CROISÉE (GROUP K-FOLD) 🚀")
n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

# Ce tableau va stocker les probabilités "pures" de tout le dataset
oof_preds_brutes = np.zeros(len(X_all))

fold = 1
for train_idx, val_idx in gkf.split(X_all, y_all, groups=groups_all):
    print(f"--- Entraînement du Fold {fold}/{n_splits} ---")
    
    X_train, y_train = X_all.iloc[train_idx], y_all.iloc[train_idx]
    X_val = X_all.iloc[val_idx]
    
    # 🏆 Tes paramètres champions (Trial 28)
    model_xgb = xgb.XGBClassifier(
        n_estimators= 506,
        max_depth= 5,
        learning_rate= 0.10423180025726148,
        min_child_weight= 10,
        subsample= 0.8738358835624531,
        colsample_bytree= 0.8723751225535342,
        gamma= 1.7503741438290679,
        scale_pos_weight= 3.964987197971567,
    )
    
    model_xgb.fit(X_train, y_train)
    
    # On prédit EXCLUSIVEMENT sur le bloc de validation que l'IA ne connaît pas !
    oof_preds_brutes[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    
    fold += 1

print("\n✅ Entraînement terminé. Prédictions Out-Of-Fold générées pour 100% des orages !")

print("🧠 ÉTAPE 3 : LISSAGE DES PROBABILITÉS (Orage par Orage)...")
df_temp = df_minute_all.copy()
df_temp['proba_brute'] = oof_preds_brutes

# Lissage sans faire baver les probabilités entre les orages
df_temp['proba_lissee'] = df_temp.groupby('storm_group_id')['proba_brute'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

print("\n🚀 ÉTAPE 4 : SIMULATEUR V5 SUR LE DATASET COMPLET (OUVERTURE IRRÉVOCABLE) 🚀")

DF_EVAL = df_temp.copy()
assert 'count_cg_audit' in DF_EVAL.columns, "❌ Erreur : 'count_cg_audit' manquante."
DF_EVAL['cible_ia'] = np.array(y_all)

# ==========================================
# 🌪️ LE FILTRE MÉTIER ULTIME
# ==========================================
alertes_par_orage = DF_EVAL.groupby('storm_group_id')['cible_ia'].sum()
orages_dangereux_ids = alertes_par_orage[alertes_par_orage > 0].index
DF_EVAL = DF_EVAL[DF_EVAL['storm_group_id'].isin(orages_dangereux_ids)].copy()

NOMBRE_ORAGES_TEST = DF_EVAL['storm_group_id'].nunique()
print(f"🎯 Crash-test géant lancé sur les {NOMBRE_ORAGES_TEST} orages menaçants (Train + Test combinés).")

# ==========================================
# 🛡️ MASQUES V4 (Danger et Gain)
# ==========================================
DF_EVAL['masque_danger'] = False
DF_EVAL['masque_gain'] = False

for storm_id, df_storm in DF_EVAL.groupby('storm_group_id'):
    idx_dernier_danger = df_storm['cible_ia'][::-1].idxmax()
    fin_alerte = idx_dernier_danger + 30
    
    DF_EVAL.loc[df_storm.loc[:idx_dernier_danger].index, 'masque_danger'] = True
    DF_EVAL.loc[df_storm.loc[idx_dernier_danger : fin_alerte].iloc[1:].index, 'masque_gain'] = True

# ==========================================
# 📊 LE CRASH-TEST ABSOLU
# ==========================================
objectifs_min_par_orage = [1, 5, 10, 15, 18, 20]
resultats = []
MINUTES_CONFIRMATION = 10 

seuils_a_tester = np.linspace(0.001, 0.999, 10)

for obj in objectifs_min_par_orage:
    gain_total_cible = obj * NOMBRE_ORAGES_TEST
    stats_pour_ce_gain = None
    
    for seuil in seuils_a_tester: 
        DF_EVAL['decision_brute'] = (DF_EVAL['proba_lissee'] > seuil).astype(int)
        
        DF_EVAL['decision_ia'] = DF_EVAL.groupby('storm_group_id')['decision_brute'].transform(
            lambda x: x.rolling(window=MINUTES_CONFIRMATION, min_periods=1).max()
        )
        
        gain_total = (DF_EVAL.loc[DF_EVAL['masque_gain'], 'decision_ia'] == 0).sum()
        
        if gain_total >= gain_total_cible:
            orages_mortels = 0
            orages_alertes = 0
            orages_faible_risque = 0
            
            for storm_id, df_storm in DF_EVAL.groupby('storm_group_id'):
                
                # On isole la zone de danger réel
                zone_danger = df_storm[df_storm['masque_danger'] == True]
                
                # Y a-t-il eu ouverture prématurée ?
                ouvertures_prematurees = zone_danger[zone_danger['decision_ia'] == 0]
                
                if not ouvertures_prematurees.empty:
                    # RÈGLE IRRÉVOCABLE : On audite depuis la 1ère minute d'ouverture
                    premiere_ouverture_idx = ouvertures_prematurees.index[0]
                    zone_post_ouverture = zone_danger.loc[premiere_ouverture_idx:]
                    
                    frappes_au_sol = zone_post_ouverture[zone_post_ouverture['count_cg_audit'] > 0]
                    
                    if not frappes_au_sol.empty:
                        distance_min_foudre = frappes_au_sol['last_known_dist'].min()
                        
                        if distance_min_foudre < 3:
                            orages_mortels += 1
                        elif distance_min_foudre < 10:
                            orages_alertes += 1
                        elif distance_min_foudre <= 20: 
                            orages_faible_risque += 1
                            
            stats_pour_ce_gain = {
                'Gain (min/orage)': f"{obj} min",
                'Seuil IA (%)': f"{seuil*100:.2f}%",
                'Total Gagné': gain_total,
                'Ratés (< 20km)': orages_faible_risque,
                'Alertes (< 10km)': orages_alertes,
                'MORTELS (< 3km)': f"{orages_mortels} ☠️" if orages_mortels > 0 else "0 ✅",
            }
            break 
    
    if stats_pour_ce_gain:
        resultats.append(stats_pour_ce_gain)
    else:
        resultats.append({
            'Gain (min/orage)': f"{obj} min",
            'Seuil IA (%)': 'Impossible',
            'Total Gagné': '-',
            'Ratés (< 20km)': '-',
            'Alertes (< 10km)': '-',
            'MORTELS (< 3km)': '-'
        })

df_resultats = pd.DataFrame(resultats)
print("\n" + "="*90)
print("📊 SIMULATEUR V5 OOF : LE CRASH-TEST CROSS-VALIDÉ (100% DES ORAGES) 📊")
print("="*90)
print(df_resultats.to_string(index=False))
print("="*90)

🚀 ÉTAPE 1 : CRÉATION DE LA GRILLE SUR TOUT LE DATASET 🚀
Création de la grille temporelle (Lookback: 5m | Horizon: 30m)...
Assemblage final...

🚀 ÉTAPE 2 : VALIDATION CROISÉE (GROUP K-FOLD) 🚀
--- Entraînement du Fold 1/5 ---
--- Entraînement du Fold 2/5 ---
--- Entraînement du Fold 3/5 ---
--- Entraînement du Fold 4/5 ---
--- Entraînement du Fold 5/5 ---

✅ Entraînement terminé. Prédictions Out-Of-Fold générées pour 100% des orages !
🧠 ÉTAPE 3 : LISSAGE DES PROBABILITÉS (Orage par Orage)...

🚀 ÉTAPE 4 : SIMULATEUR V5 SUR LE DATASET COMPLET (OUVERTURE IRRÉVOCABLE) 🚀
🎯 Crash-test géant lancé sur les 2400 orages menaçants (Train + Test combinés).

📊 SIMULATEUR V5 OOF : LE CRASH-TEST CROSS-VALIDÉ (100% DES ORAGES) 📊
Gain (min/orage) Seuil IA (%)  Total Gagné  Ratés (< 20km)  Alertes (< 10km) MORTELS (< 3km)
           1 min       11.19%        16864               9                 4            1 ☠️
           5 min       11.19%        16864               9                 4            1 ☠️


Optimisation optuna (pas besoin de le lancer pour testé le modèle)

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import GroupKFold
import warnings

warnings.filterwarnings('ignore')

print("🚀 Lancement d'Optuna : OBJECTIF 'ZÉRO MORTEL' (< 3km) 🚀")

def objective(trial):
    # 1. Hyperparamètres à tester
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 400, 800),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.2, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),
        'gamma': trial.suggest_float('gamma', 1.0, 4.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 3.0, 6.0), 
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 2. Validation Croisée OOF (5 Folds)
    gkf = GroupKFold(n_splits=5)
    oof_preds = np.zeros(len(X_all))
    
    for train_idx, val_idx in gkf.split(X_all, y_all, groups=groups_all):
        X_train, y_train = X_all.iloc[train_idx], y_all.iloc[train_idx]
        X_val = X_all.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train)
        
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

    # 3. Lissage OOF
    df_eval = DF_EVAL.copy() # On utilise ton DF_EVAL qui a déjà les masques
    df_eval['proba_brute'] = oof_preds
    df_eval['proba_lissee'] = df_eval.groupby('storm_group_id')['proba_brute'].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    
    # 4. MINI-SIMULATEUR CIBLÉ SUR < 3KM
    MINUTES_CONFIRMATION = 10
    meilleur_gain_modele = -1000 
    
    # Comme la règle est plus permissive (<3km), l'IA peut monter plus haut en seuil
    # On scanne de 0.1% à 30% pour trouver la limite mortelle
    seuils_rapides = np.linspace(0.001, 0.30, 40)
    
    for seuil in seuils_rapides:
        df_eval['decision_brute'] = (df_eval['proba_lissee'] > seuil).astype(int)
        
        df_eval['decision_ia'] = df_eval.groupby('storm_group_id')['decision_brute'].transform(
            lambda x: x.rolling(window=MINUTES_CONFIRMATION, min_periods=1).max()
        )
        
        gain = (df_eval.loc[df_eval['masque_gain'], 'decision_ia'] == 0).sum()
        
        orages_mortels = 0
        zone_danger = df_eval[df_eval['masque_danger'] == True]
        ouvertures_prematurees = zone_danger[df_eval.loc[zone_danger.index, 'decision_ia'] == 0]
        
        # S'il y a eu des ouvertures pendant le danger, on audite
        if not ouvertures_prematurees.empty:
            orages_ouverts = ouvertures_prematurees['storm_group_id'].unique()
            
            for storm_id in orages_ouverts:
                df_storm = df_eval[(df_eval['storm_group_id'] == storm_id) & (df_eval['masque_danger'] == True)]
                ouv_storm = df_storm[df_storm['decision_ia'] == 0]
                
                if not ouv_storm.empty:
                    idx_prem = ouv_storm.index[0]
                    zone_post = df_storm.loc[idx_prem:]
                    
                    frappes = zone_post[zone_post['count_cg_audit'] > 0]
                    
                    if not frappes.empty:
                        # 💡 LA RÈGLE DES 3 KM EST ICI
                        if frappes['last_known_dist'].min() < 3:
                            orages_mortels += 1
                            break # Optimisation: 1 mort suffit à invalider ce seuil, on passe au suivant
                            
        # Si on a survécu (0 mortel), on regarde si le gain bat notre record
        if orages_mortels == 0:
            if gain > meilleur_gain_modele:
                meilleur_gain_modele = gain
                
    return meilleur_gain_modele

# =========================================================
# 🚀 LANCEMENT DE L'ÉTUDE
# =========================================================
study = optuna.create_study(direction='maximize')

# On injecte ton meilleur modèle actuel pour l'aider à démarrer
study.enqueue_trial({
    'n_estimators': 506,
    'max_depth': 5,
    'learning_rate': 0.10423180025726148,
    'min_child_weight': 10,
    'subsample': 0.8738358835624531,
    'colsample_bytree': 0.8723751225535342,
    'gamma': 1.7503741438290679,
    'scale_pos_weight': 3.964987197971567,
    'random_state':42,
    'n_jobs':-1
})

study.optimize(objective, n_trials=30)

print("\n🏆 OPTIMISATION TERMINÉE (CIBLE < 3KM) 🏆")
print(f"Minutes OOF garanties (0 crash mortel) : {study.best_value}")
for key, value in study.best_params.items():
    print(f"    '{key}': {value},")

[I 2026-03-27 12:17:06,230] A new study created in memory with name: no-name-46e92388-578c-4d4f-8089-9d5ba36bb758


🚀 Lancement d'Optuna : OBJECTIF 'ZÉRO MORTEL' (< 3km) 🚀


[I 2026-03-27 12:18:47,114] Trial 0 finished with value: 25160.0 and parameters: {'n_estimators': 506, 'max_depth': 5, 'learning_rate': 0.10423180025726148, 'min_child_weight': 10, 'subsample': 0.8738358835624531, 'colsample_bytree': 0.8723751225535342, 'gamma': 1.7503741438290679, 'scale_pos_weight': 3.964987197971567}. Best is trial 0 with value: 25160.0.
[I 2026-03-27 12:20:21,620] Trial 1 finished with value: 18081.0 and parameters: {'n_estimators': 606, 'max_depth': 6, 'learning_rate': 0.05428751661630012, 'min_child_weight': 8, 'subsample': 0.9488440759450114, 'colsample_bytree': 0.8949812345476832, 'gamma': 1.866465811322776, 'scale_pos_weight': 4.552171219199056}. Best is trial 0 with value: 25160.0.
[I 2026-03-27 12:22:04,186] Trial 2 finished with value: 25802.0 and parameters: {'n_estimators': 565, 'max_depth': 4, 'learning_rate': 0.05071177983347182, 'min_child_weight': 5, 'subsample': 0.8690456293781198, 'colsample_bytree': 0.9339537206585246, 'gamma': 1.691988033125926, '

Evaluation du modèle

In [ ]:
import numpy as np
import pandas as pd

# ==================================================
# ⚙️ CONFIGURATION DE L'ÉVALUATION
# ==================================================
DIST_AUDIT_KM = RISQUE_KM 
MINUTES_CONFIRMATION = 10 

print(f"🔍 SCANNER OOF : RECHERCHE DU SEUIL PARFAIT (Audit Jury: {DIST_AUDIT_KM}km | Règle Humaine: {ZONE}km) 🔍")

# Vérification des colonnes essentielles
assert 'count_cg_audit' in DF_EVAL.columns, "Il manque count_cg_audit dans DF_EVAL."
assert 'count_cg_zonekm' in DF_EVAL.columns, "Il manque count_cg_zonekm dans DF_EVAL. (Modifiez creer_grille_temporelle pour l'ajouter !)"

# ==================================================
# 📏 CALCUL DU DÉNOMINATEUR RÉEL (N_L3)
# ==================================================
N_L3 = DF_EVAL['count_cg_audit'].sum()
print(f"⚡ Nombre total d'éclairs critiques (< {DIST_AUDIT_KM}km) (N_L3) : {N_L3}")

RISQUE_TOLERE = RISQUE_POURCENT 
seuils_a_tester = np.linspace(0.01, 0.60, 500)

meilleur_seuil_jury = 0
max_gain_total_jury = -1
risque_final = -1

# Variables pour stocker les statistiques par orage du meilleur seuil
stats_gains_par_orage = None

print(f"Scannage de {len(seuils_a_tester)} seuils en cours...")

for seuil in seuils_a_tester:
    # 1. Décision brute et Lissage de confirmation (10 minutes)
    decision_brute = (DF_EVAL['proba_lissee'] > seuil).astype(int)
    decision_ia = decision_brute.groupby(DF_EVAL['storm_group_id']).transform(
        lambda x: x.rolling(window=MINUTES_CONFIRMATION, min_periods=1).max()
    )
    
    # ==========================================
    # 🚨 LA RÈGLE IRRÉVOCABLE ET L'ALERTE HUMAINE
    # ==========================================
    # A. Le déclencheur humain officiel (Fixe à 20 km, immuable !)
    eclair_critique_humain = (DF_EVAL['count_cg_zonekm'] > 0).astype(int)
    alerte_declenchee = eclair_critique_humain.groupby(DF_EVAL['storm_group_id']).cummax()
    
    # B. L'IA décide d'Ouvrir (0) UNIQUEMENT si l'alerte officielle a bien sonné la minute d'avant
    alerte_precedente = alerte_declenchee.groupby(DF_EVAL['storm_group_id']).shift(1).fillna(0)
    ordre_ouverture_ia = (alerte_precedente == 1) & (decision_ia == 0)
    
    # C. Verrou Irrévocable : dès qu'elle ouvre pendant l'alerte, ça reste ouvert
    est_ouvert_definitivement = ordre_ouverture_ia.groupby(DF_EVAL['storm_group_id']).cummax()
    
    # D. L'état final du tarmac IA : Fermé (1) SEULEMENT si alerte en cours ET pas rouvert
    decision_irrevocable = ((alerte_declenchee == 1) & (~est_ouvert_definitivement)).astype(int)
    
    # ==========================================
    # 👨‍✈️ LA DÉCISION DE L'HUMAIN (La Vraie Baseline)
    # ==========================================
    # L'humain ferme pendant EXACTEMENT 30 minutes après CHAQUE éclair <= 20 km
    etat_humain_ferme = eclair_critique_humain.groupby(DF_EVAL['storm_group_id']).transform(
        lambda x: x.rolling(window=30, min_periods=1).max()
    )

    # ==========================================
    # ⏱️ CALCUL DU GAIN PUR 
    # ==========================================
    # Gain = L'humain a peur et ferme (1), MAIS l'IA est intelligente et a ouvert (0)
    minutes_gagnees = (etat_humain_ferme == 1) & (decision_irrevocable == 0) & (alerte_declenchee == 1)
    gain_total = minutes_gagnees.sum()
    
    # ==========================================
    # ⚖️ AUDIT DU RISQUE (JURY)
    # ==========================================
    # L'audit du jury s'applique avec SA distance (DIST_AUDIT_KM) via count_cg_audit
    ouvertures_pendant_danger = (decision_irrevocable == 0) & (alerte_declenchee == 1)
    M_L3 = DF_EVAL.loc[ouvertures_pendant_danger, 'count_cg_audit'].sum()

    # Calcul du Risque R
    R = M_L3 / N_L3 if N_L3 > 0 else 0
    
    if R <= RISQUE_TOLERE:
        if gain_total > max_gain_total_jury:
            max_gain_total_jury = gain_total
            meilleur_seuil_jury = seuil
            risque_final = R
            
            # Capture des gains par orage (Statistiques)
            stats_gains_par_orage = minutes_gagnees.groupby(DF_EVAL['storm_group_id']).sum()

print("\n" + "="*80)
print("🏆 VERDICT DU SCANNER OOF (MÉTRIQUE OFFICIELLE DU JURY) 🏆")
print("="*80)
if meilleur_seuil_jury > 0:
    print(f"⚖️  OBJECTIF 'JURY' (Risque < 2% sur les éclairs à < {DIST_AUDIT_KM}km) :")
    print(f"   👉 Seuil Parfait : {meilleur_seuil_jury*100:.3f}%")
    print(f"   ✅ Gain Total    : {max_gain_total_jury} minutes")
    print(f"   ⚡ Risque final  : {risque_final*100:.2f}% (M_L3 / N_L3)")
    
    if stats_gains_par_orage is not None:
        print("-" * 80)
        print(f"📊 DISTRIBUTION DU GAIN PAR ORAGE (pour ce seuil) :")
        print(f"   🔹 Moyenne       : {stats_gains_par_orage.mean():.1f} minutes / orage")
        print(f"   🔹 Médiane       : {stats_gains_par_orage.median():.1f} minutes / orage")
        print(f"   🔹 Écart-type    : {stats_gains_par_orage.std():.1f} minutes")
        print(f"   🔹 Gain Max      : {stats_gains_par_orage.max():.1f} minutes")
        print(f"   🔹 Gain Min      : {stats_gains_par_orage.min():.1f} minutes")
else:
    print("⚠️ Aucun seuil n'a respecté la limite de risque de 2%.")

Analyse des variable importante

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 ANALYSE DE L'IMPORTANCE DES VARIABLES (XGBOOST GAIN) 📊")

# 1. Extraction des importances du modèle
# L'attribut feature_importances_ de XGBoost donne l'importance par "Gain"
importances = model_xgb.feature_importances_

# 2. Création d'un DataFrame pour trier tout ça proprement
df_importance = pd.DataFrame({
    'Feature': features_a_entrainer,
    'Importance': importances
})

# 3. Tri des variables de la plus importante à la moins importante
df_importance = df_importance.sort_values(by='Importance', ascending=False)

# 4. Affichage du Top 10 dans la console
print("\n🏆 Top 10 des variables les plus importantes :")
print(df_importance.head(10).to_string(index=False))

# 5. Visualisation Graphique (Bar Chart)
plt.figure(figsize=(12, 10))
sns.barplot(x='Importance', y='Feature', data=df_importance, palette='viridis')

plt.title("Importance des Variables (XGBoost - Gain)", fontsize=16, fontweight='bold')
plt.xlabel("Niveau d'importance (Contribution à la décision)", fontsize=12)
plt.ylabel("Variables", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 6. Variables inutiles (Importance = 0)
variables_zero = df_importance[df_importance['Importance'] == 0]['Feature'].tolist()
if len(variables_zero) > 0:
    print(f"\n⚠️ Attention, ces {len(variables_zero)} variables ont une importance de 0 (à retirer potentiellement) :")
    print(variables_zero)

Entrainement finale du modèle sur le jeu d'entrainement

In [ ]:
# 1. Préparation des données totales (Train + Validation)
# Utilise le DataFrame que tu as déjà passé dans creer_grille_temporelle
X_total = df_minute_all[features_a_entrainer]
y_total = df_minute_all['target_danger_30m'].squeeze()

# 2. Initialisation avec tes meilleurs paramètres
model_final = xgb.XGBClassifier(
    n_estimators=506,
    max_depth=5,
    learning_rate=0.10423180025726148,
    min_child_weight=10,
    subsample=0.8738358835624531,
    colsample_bytree=0.8723751225535342,
    gamma=1.7503741438290679,
    scale_pos_weight=3.964987197971567,
    random_state=42,
    n_jobs=-1
)

# 3. Entraînement sur la TOTALITÉ du dataset
print("🏋️ Entraînement final sur 100% des données...")
model_final.fit(X_total, y_total)
print("✅ Modèle prêt pour le dataset d'évaluation !")

Chargement du dataset de test

In [ ]:
import pandas as pd
import numpy as np

# 1. Chargement du nouveau dataset
df_work = pd.read_csv('../data/segment_alerts_all_airports_test.csv')


Traitement des données

In [ ]:
FEATURES = [
    # 📍 1. Spatiales & Localisation
    'lon', 'lat', 'maxis', 'dist', 'azimuth', 

    # ⏱️ 2. Temporelles (Absolues et Cycliques)
    'time_since_last_strike', 'time_since_storm_start',
    'time_since_last_N', 
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos',

    # 🏎️ 3. Cinématiques (Mouvement de l'orage)
    'delta_dist', 'delta_azimuth', 'delta_azimuth_norm', 'speed', 'acceleration',

    # ⚡ 4. Intensité & Type
    'abs_amplitude', 'polarity',

    # 📈 5. Ratios, Comptages & Activité immédiate
    'strikes_last_5min', 'strikes_last_10min', 'strikes_last_20min', 'activity_trend',
    'cum_n_strikes', 'cum_cg_strikes', 'ratio_n_r_cumul',
    'cumulative_count', 

    # 📚 6. Statistiques Cumulées (Historique de l'orage jusqu'à l'instant T)
    'mean_dist_so_far', 'std_dist_so_far', 'mean_amp_so_far',

    # 🔄 7. Tendances à Court Terme (Rolling Features)
    'rolling_icloud_mean', 'rolling_amp_mean', 'rolling_delta_dist_sum',
    'rolling_time_diff_mean', 'rolling_min_dist', 'rolling_dist_std',
    'rolling_azimuth_std', 

    # 🌍 8. Comparaisons Globales (Storm-level vs instant T)
    'amp_vs_storm_mean', 'amp_vs_storm_max',
    'storm_max_amplitude', 

    # 🎯 --- CATÉGORIQUE / EMBEDDING (OBLIGATOIREMENT À LA TOUTE FIN) ---
    'airport_code'
]

import pandas as pd
import numpy as np


# =========================================================
# CRÉATION DES GROUPES D'ORAGES (SÉQUENCES ET CONTEXTE) - VERSION OFFICIELLE
# =========================================================
def add_storm_groups_by_target(df, context_size=5, time_window_minutes=30):
    """
    Crée les groupes d'orages en se basant STRICTEMENT sur la colonne de cible officielle.
    Modifie le DataFrame `df` EN PLACE.
    """
    # Sécurité temporelle
    if not pd.api.types.is_datetime64_any_dtype(df['date']):
        df['date'] = pd.to_datetime(df['date'])

    # Tri chronologique et réinitialisation de l'index EN PLACE
    df.sort_values(['airport', 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Initialisation à -1 (Bruit par défaut)
    df['storm_group_id'] = -1

    # On repère où sont les cibles (les points de repère de l'alerte)
    is_target = df['is_last_lightning_cloud_ground'].notna()

    if not is_target.any():
        print("⚠️ Aucun groupe valide trouvé. Toutes les lignes sont à -1.")
        return

    is_true = df['is_last_lightning_cloud_ground'].isin([True, 1, 1.0, 'True'])

    # Marquer le début des séquences "brutes"
    airport_changed = df['airport'] != df['airport'].shift(1)
    after_true = is_true.shift(1).fillna(False)
    starts_new_group = airport_changed | after_true

    # On crée un ID brut pour toute la séquence (y compris le bruit du début)
    raw_group_id = starts_new_group.cumsum()

    valid_indices = []

    # On parcourt chaque séquence brute
    for grp_id, group_df in df.groupby(raw_group_id):
        targets = group_df['is_last_lightning_cloud_ground'].notna()

        if not targets.any():
            continue  # Pas de cible dans cette séquence brute

        # Index et heure du tout premier déclenchement de l'alerte
        first_target_idx = targets.idxmax()
        first_target_time = group_df.loc[first_target_idx, 'date']

        # 🚀 CORRECTION DU BUG DE L'INDEX -1 :
        if first_target_idx == group_df.index[0]:
            before_df = pd.DataFrame(columns=group_df.columns)
        else:
            before_df = group_df.loc[:first_target_idx - 1]

        # Filtre temporel : Uniquement dans la fenêtre demandée (ex: 30 min)
        if not before_df.empty:
            time_diffs = first_target_time - before_df['date']
            valid_context = before_df[time_diffs <= pd.Timedelta(minutes=time_window_minutes)]
            context_indices = valid_context.index[-context_size:].tolist()
        else:
            context_indices = []

        # Les index de l'alerte officielle (jusqu'à la fin du groupe)
        alert_indices = group_df.loc[first_target_idx:].index.tolist()

        # On rassemble les index validés
        valid_indices.extend(context_indices + alert_indices)

    # On assigne les vrais IDs uniquement aux lignes validées
    df.loc[valid_indices, 'storm_group_id'] = raw_group_id[valid_indices]

    # Renumérotation propre
    valid_mask = df['storm_group_id'] != -1
    if valid_mask.any():
        df.loc[valid_mask, 'storm_group_id'] = pd.factorize(df.loc[valid_mask, 'storm_group_id'])[0] + 1

    print("✅ Groupes d'orages restaurés selon la cible officielle du Dataset !")


# =========================================================
# CALCUL DE LA CIBLE (TIME TO END)
# =========================================================
def add_time_to_end_target(df):
    """
    Calcule le temps restant (en minutes) avant la levée de l'alerte.
    Note Anti-Leakage : Il est normal et OBLIGATOIRE que cette fonction regarde
    dans le futur (bfill), car elle calcule la Target (Y) que l'IA devra deviner.
    """
    print("⏳ Calcul de la cible de Régression (RUL) par aéroport...")

    df['date'] = pd.to_datetime(df['date'])
    df.sort_values(by=['airport', 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Identifier l'heure exacte des éclairs qui marquent la fin
    df['next_true_time'] = df['date'].where(df['is_last_lightning_cloud_ground'].isin([True, 1, 1.0, 'True']))

    # Remplir vers le haut (bfill) EN GROUPANT PAR ORAGE
    df['next_true_time'] = df.groupby('storm_group_id')['next_true_time'].bfill()

    # Calcul de l'écart temporel en minutes + 30 minutes réglementaires
    df['time_to_end'] = (df['next_true_time'] - df['date']).dt.total_seconds() / 60.0 + 30.0

    # 🚀 CORRECTION : Suppression de la ligne qui mettait des NaN sur les éclairs intra-nuageux.
    # Un réseau de neurones crashe si sa Target contient des NaN !

    # Nettoyage
    df.drop(columns=['next_true_time'], inplace=True)

    if 'storm_group_id' in df.columns:
        df.sort_values(by=['storm_group_id', 'date'], inplace=True)
        df.reset_index(drop=True, inplace=True)


# =========================================================
# FILTRAGE DES ORAGES UTILES
# =========================================================
def filter_useful_storms(df):
    """
    Garde uniquement les orages qui ont au moins un éclair au sol (CG)
    à moins de 20 km (les orages qui justifient vraiment une alerte).
    """
    initial_rows = len(df)
    initial_groups = df['storm_group_id'].nunique()

    # Identification des groupes valides
    is_alert_strike = (df['icloud'] == 0) & (df['dist'] <= ZONE)
    valid_group_ids = df.loc[is_alert_strike, 'storm_group_id'].unique()

    # Filtrage IN-PLACE
    mask_to_keep = df['storm_group_id'].isin(valid_group_ids)
    final_rows = mask_to_keep.sum()
    final_groups = len(valid_group_ids)

    df.drop(df[~mask_to_keep].index, inplace=True)

    # Réarrangement des IDs
    df['storm_group_id'] = pd.factorize(df['storm_group_id'])[0] + 1
    df.reset_index(drop=True, inplace=True)

    deleted_rows = initial_rows - final_rows
    deleted_groups = initial_groups - final_groups

    print("-" * 40)
    print("⚡ RAPPORT DE FILTRAGE DES ORAGES ⚡")
    print("-" * 40)
    print(f"Groupes : {initial_groups} -> {final_groups} (-{deleted_groups})")
    print(f"Lignes  : {initial_rows} -> {final_rows} (-{deleted_rows})")
    print("-" * 40)


# =========================================================
# NETTOYAGE DES ANOMALIES CAPTEURS
# =========================================================
def remove_pise_2016(df):
    """
    Supprime les données de l'aéroport de Pise pour l'année 2016
    (anomalie capteur Météorage).
    """
    if not pd.api.types.is_datetime64_any_dtype(df['date']):
        df['date'] = pd.to_datetime(df['date'])

    mask_to_drop = (df['airport'] == 'Pise') & (df['date'].dt.year == 2016)
    lignes_a_supprimer = mask_to_drop.sum()

    if lignes_a_supprimer > 0:
        df.drop(df[mask_to_drop].index, inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f"🧹 NETTOYAGE : Suppression de {lignes_a_supprimer} lignes pour Pise (2016).")
    else:
        print("✅ Aucune donnée de Pise 2016 n'a été trouvée/supprimée.")


# =========================================================
# SUPPRESSION DU BRUIT (HORS SÉQUENCES)
# =========================================================
def remove_noise_data(df):
    """
    Supprime toutes les lignes qui n'appartiennent pas à un groupe valide (-1).
    """
    initial_rows = len(df)
    mask_noise = df['storm_group_id'] == -1
    rows_to_drop = mask_noise.sum()

    if rows_to_drop > 0:
        df.drop(df[mask_noise].index, inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f"🧹 NETTOYAGE : Bruit supprimé ({rows_to_drop} lignes). Lignes restantes : {len(df)}")
    else:
        print("✅ Aucun bruit trouvé.")


# =========================================================
# CRÉATION DES FEATURES SPATIALES
# =========================================================
def add_zone_features(df,ZONE):
    """Indique si l'éclair a frappé dans la zone critique des 20km."""
    df['is_in_20km'] = (df['dist'] <= ZONE).astype(int)


# =========================================================
# FORMATAGE : IDENTIFIANTS D'ALERTE
# =========================================================
def format_alert_id(df):
    df['airport_alert_id'] = df['airport_alert_id'].fillna(0).astype(int)


# =========================================================
# FORMATAGE : CIBLE FIN D'ALERTE
# =========================================================
def format_last_lightning(df):
    """
    ⚠️ Ne lancer cette fonction qu'APRÈS la création des storm_groups !
    """
    df['is_last_lightning_cloud_ground'] = df['is_last_lightning_cloud_ground'].fillna(False).astype(int)


# =========================================================
# FORMATAGE : TYPE D'ÉCLAIR (INTRA-NUAGEUX / SOL)
# =========================================================
def format_icloud(df):
    df['icloud'] = df['icloud'].astype(int)


# =========================================================
# FORMATAGE : DATES
# =========================================================
def format_date(df):
    df['date'] = pd.to_datetime(df['date'])

import numpy as np
import pandas as pd

STORM_GROUP_COL = 'storm_group_id'


# =========================================================
# PRÉPARATION : TRI DES SÉQUENCES
# =========================================================
def sort_for_sequences(df):
    """Trie le dataset chronologiquement par orage."""
    df.sort_values([STORM_GROUP_COL, 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)


# =========================================================
# FEATURES TEMPORELLES
# =========================================================
def add_temporal_features(df):
    """Calcule les écarts de temps entre éclairs et depuis le début de l'orage."""
    df['time_since_last_strike'] = df.groupby(STORM_GROUP_COL)['date'].diff().dt.total_seconds().fillna(0)

    # transform('min') est safe ici car le début de l'orage est un point de repère fixe dans le passé
    min_dates = df.groupby(STORM_GROUP_COL)['date'].transform('min')
    df['time_since_storm_start'] = (df['date'] - min_dates).dt.total_seconds() / 60.0


# =========================================================
# FEATURES CINÉMATIQUES (MOUVEMENT)
# =========================================================
def add_kinematic_features(df):
    """Calcule les déplacements (distance et angle) entre éclairs consécutifs."""
    df['delta_dist'] = df.groupby(STORM_GROUP_COL)['dist'].diff().fillna(0)
    df['delta_azimuth'] = df.groupby(STORM_GROUP_COL)['azimuth'].diff().fillna(0)


# =========================================================
# FEATURES D'INTENSITÉ (AMPLITUDE)
# =========================================================
def add_intensity_features(df):
    """Extrait l'amplitude absolue et la polarité de l'éclair."""
    df['abs_amplitude'] = df['amplitude'].abs()
    df['polarity'] = np.sign(df['amplitude']).astype(int)


# =========================================================
# FEATURES CUMULÉES (HISTORIQUE DE L'ORAGE)
# =========================================================
def add_cumulative_features(df):
    """
    Calcule les statistiques cumulées depuis le début de l'orage jusqu'à l'éclair actuel.
    L'utilisation de .expanding() garantit l'absence de leakage vers le futur.
    """
    df.sort_values([STORM_GROUP_COL, 'date'], inplace=True)
    df['storm_duration_so_far'] = df['time_since_storm_start']
    df['cumulative_count'] = df.groupby(STORM_GROUP_COL).cumcount() + 1

    df['mean_dist_so_far'] = df.groupby(STORM_GROUP_COL)['dist'].expanding().mean().reset_index(level=0, drop=True)
    df['mean_amp_so_far'] = df.groupby(STORM_GROUP_COL)['amplitude'].expanding().mean().reset_index(level=0, drop=True)

    df['std_dist_so_far'] = df.groupby(STORM_GROUP_COL)['dist'].expanding().std().reset_index(level=0, drop=True)
    df['std_dist_so_far'] = df['std_dist_so_far'].fillna(0.0)
    print("✅ Cumulative features ajoutées (sans leakage)")


# =========================================================
# INCERTITUDE SPATIALE
# =========================================================
def add_uncertainty_features(df):
    """Estime l'aire d'incertitude de localisation de l'éclair."""
    df['error_area_km2'] = np.pi * (df['maxis'] ** 2)


# =========================================================
# ROLLING FEATURES (TENDANCES À COURT TERME)
# =========================================================
def add_rolling_features(df, length):
    """Calcule les moyennes glissantes sur les N derniers éclairs."""
    print("Calcul des Rolling Features (Tendances)...")
    df['rolling_icloud_mean'] = df.groupby(STORM_GROUP_COL)['icloud'].transform(
        lambda x: x.rolling(window=length, min_periods=1).mean()
    )
    df['rolling_amp_mean'] = df.groupby(STORM_GROUP_COL)['abs_amplitude'].transform(
        lambda x: x.rolling(window=length, min_periods=1).mean()
    )
    df['rolling_delta_dist_sum'] = df.groupby(STORM_GROUP_COL)['delta_dist'].transform(
        lambda x: x.rolling(window=length, min_periods=1).sum()
    )
    df['rolling_time_diff_mean'] = df.groupby(STORM_GROUP_COL)['time_since_last_strike'].transform(
        lambda x: x.rolling(window=length, min_periods=1).mean()
    )


# =========================================================
# ENCODAGE CYCLIQUE DU TEMPS
# =========================================================
def add_cyclical_time_features(df):
    """Transforme l'heure et la date en coordonnées circulaires (sin/cos)."""
    month = df['date'].dt.month
    hour = df['date'].dt.hour
    day_of_year = df['date'].dt.dayofyear

    df['month_sin'] = np.sin(2 * np.pi * month / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * month / 12.0)
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24.0)
    df['day_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    df['day_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)


# =========================================================
# ACTIVITÉ RÉCENTE (STRIKE RATES)
# =========================================================
def add_activity_rate_features(df):
    """Compte le nombre d'éclairs dans les X dernières minutes strictes (closed='left')."""
    df['date_for_rolling'] = df['date']

    def count_strikes_last_n_min(group, minutes):
        group = group.set_index('date_for_rolling')
        return group['icloud'].rolling(f'{minutes}min', closed='left').count().values

    for minutes in [5, 10, 20]:
        df[f'strikes_last_{minutes}min'] = df.groupby(STORM_GROUP_COL, group_keys=False).apply(
            lambda g: pd.Series(count_strikes_last_n_min(g, minutes), index=g.index)
        )

    df.drop(columns=['date_for_rolling'], inplace=True)
    df['activity_trend'] = df['strikes_last_5min'] / (df['strikes_last_10min'] + 1e-6)


# =========================================================
# STABILITÉ SPATIALE
# =========================================================
def add_spatial_stability_features(df, length=5):
    """Mesure la dispersion géographique récente de l'orage."""
    df['rolling_min_dist'] = df.groupby(STORM_GROUP_COL)['dist'].transform(
        lambda x: x.rolling(window=length, min_periods=1).min()
    )
    df['rolling_dist_std'] = df.groupby(STORM_GROUP_COL)['dist'].transform(
        lambda x: x.rolling(window=length, min_periods=1).std().fillna(0)
    )
    df['rolling_azimuth_std'] = df.groupby(STORM_GROUP_COL)['azimuth'].transform(
        lambda x: x.rolling(window=length, min_periods=1).std().fillna(0)
    )


# =========================================================
# RATIOS DE TYPES D'ÉCLAIRS (N vs CG)
# =========================================================
def add_ratio_features(df, length):
    """Calcule la proportion d'éclairs intra-nuageux par rapport aux éclairs au sol."""
    df['cum_n_strikes'] = (df['icloud'] == 1).astype(int).groupby(df[STORM_GROUP_COL]).cumsum()
    df['cum_cg_strikes'] = (df['icloud'] == 0).astype(int).groupby(df[STORM_GROUP_COL]).cumsum()

    df['ratio_n_r_cumul'] = df['cum_n_strikes'] / (df['cum_cg_strikes'] + 1e-6)
    df['rolling_n_ratio'] = df.groupby(STORM_GROUP_COL)['icloud'].transform(
        lambda x: x.rolling(window=length, min_periods=1).mean()
    )


# =========================================================
# VITESSE ET ACCÉLÉRATION DE L'ORAGE
# =========================================================
def add_velocity_features(df):
    """Estime la vitesse apparente de rapprochement/éloignement."""
    df['speed'] = df['delta_dist'] / (df['time_since_last_strike'] / 60.0 + 1e-6)
    df['speed'] = df['speed'].replace([np.inf, -np.inf], 0.0)  # Sécurité anti-crash

    df['acceleration'] = df.groupby(STORM_GROUP_COL)['speed'].diff().fillna(0)
    df['delta_azimuth_norm'] = (df['delta_azimuth'] + 180) % 360 - 180


# =========================================================
# AMPLITUDE RELATIVE AU PASSÉ
# =========================================================
def add_relative_amplitude_features(df):
    """Compare l'amplitude de l'éclair actuel à la moyenne/max historique de l'orage."""
    df.sort_values([STORM_GROUP_COL, 'date'], inplace=True)

    df['expanding_amp_mean'] = df.groupby(STORM_GROUP_COL)['amplitude'].expanding().mean().reset_index(level=0,
                                                                                                       drop=True)
    df['expanding_amp_max'] = df.groupby(STORM_GROUP_COL)['abs_amplitude'].expanding().max().reset_index(level=0,drop=True)

    df['amp_vs_storm_mean'] = df['amplitude'] / df['expanding_amp_mean'].replace(0, np.nan)
    df['amp_vs_storm_max'] = df['abs_amplitude'] / df['expanding_amp_max'].replace(0, np.nan)

    df.drop(columns=['expanding_amp_mean', 'expanding_amp_max'], inplace=True)
    df['amp_vs_storm_mean'] = df['amp_vs_storm_mean'].fillna(1.0)
    df['amp_vs_storm_max'] = df['amp_vs_storm_max'].fillna(1.0)
    print("✅ Relative amplitude features ajoutées (sans leakage)")


# =========================================================
# POSITION TEMPORELLE DE L'ÉCLAIR
# =========================================================
def add_position_features(df):
    """Position ordinale dans la séquence et temps depuis le dernier intra-nuageux (N)."""
    df['position_in_storm'] = df.groupby(STORM_GROUP_COL).cumcount()

    # Marquer la date des N uniquement
    is_N = (df['icloud'] == 1)
    df['_last_N_date'] = df['date'].where(is_N)

    # Forward fill PAR ORAGE — propage à tous les éclairs suivants (R et N) sans fuite
    df['_last_N_date'] = df.groupby(STORM_GROUP_COL)['_last_N_date'].ffill()

    # Calcul du temps écoulé depuis ce dernier N
    df['time_since_last_N'] = (df['date'] - df['_last_N_date']).dt.total_seconds() / 60.0

    # -1 = pas encore de N dans cet orage
    df['time_since_last_N'] = df['time_since_last_N'].fillna(-1)

    df.drop(columns=['_last_N_date'], inplace=True)
    print("✅ time_since_last_N ajouté (propagé à tous les éclairs)")


# =========================================================
# SURVIE ET DANGEROSITÉ IMMÉDIATE (DERNIER R)
# =========================================================
def add_survival_features(df):
    """
    Temps depuis le dernier éclair sol (R) dans la zone d'alerte (<20km).
    Essentiel pour estimer si l'orage est en train de se dissiper.
    """
    # Marquer la date des R uniquement
    is_R = (df['icloud'] == 0) & (df['dist'] <= ZONE)
    df['_last_R_date'] = df['date'].where(is_R)

    # Forward fill PAR ORAGE
    df['_last_R_date'] = df.groupby(STORM_GROUP_COL)['_last_R_date'].ffill()

    # Calcul du temps écoulé depuis ce dernier R
    df['time_since_last_R'] = (df['date'] - df['_last_R_date']).dt.total_seconds() / 60.0

    # -1 = pas encore de R dans cet orage (contexte N avant le premier R)
    df['time_since_last_R'] = df['time_since_last_R'].fillna(-1)

    df.drop(columns=['_last_R_date'], inplace=True)
    print("✅ time_since_last_R ajouté (propagé à tous les éclairs)")


# =========================================================
# STATISTIQUES GLOBALES DE L'ORAGE (SO FAR)
# =========================================================
def add_storm_level_features(df):
    """Caractéristiques globales de l'orage jusqu'à l'instant T (sans lire le futur)."""
    df.sort_values([STORM_GROUP_COL, 'date'], inplace=True)
    df['storm_total_lightnings'] = df.groupby(STORM_GROUP_COL).cumcount() + 1

    df['_icloud_cumsum'] = df.groupby(STORM_GROUP_COL)['icloud'].cumsum()
    df['storm_cloud_ratio'] = df['_icloud_cumsum'] / df['storm_total_lightnings']
    df.drop(columns=['_icloud_cumsum'], inplace=True)

    df['storm_mean_amplitude'] = df.groupby(STORM_GROUP_COL)['amplitude'].expanding().mean().reset_index(level=0,
                                                                                                         drop=True)
    df['storm_max_amplitude'] = df.groupby(STORM_GROUP_COL)['abs_amplitude'].expanding().max().reset_index(level=0,
                                                                                                           drop=True)
    df['storm_mean_dist'] = df.groupby(STORM_GROUP_COL)['dist'].expanding().mean().reset_index(level=0, drop=True)

    print("✅ Storm-level features ajoutées (sans leakage)")


# =========================================================
# EMBEDDING AÉROPORT
# =========================================================
def add_airport_code(df):
    """Encode le nom de l'aéroport en entier pour l'Embedding PyTorch."""
    mapping = {'Bron': 0, 'Bastia': 1, 'Ajaccio': 2, 'Nantes': 3, 'Pise': 4, 'Biarritz': 5}
    df['airport_code'] = df['airport'].map(mapping)
    print("✅ Codes aéroports ajoutés pour l'Embedding")


# =========================================================
# NETTOYAGE FINAL DES NAN
# =========================================================
def fill_nan_features(df):
    """Remplace les valeurs manquantes générées par les calculs glissants/cumulés."""
    fills = {
        'strikes_last_5min': 0,
        'strikes_last_10min': 0,
        'strikes_last_20min': 0,
        'activity_trend': 1.0,
        'rolling_min_dist': df['dist'] if 'dist' in df.columns else 0,  # Utilise la dist actuelle si pas d'historique
        'rolling_dist_std': 0,
        'rolling_azimuth_std': 0,
        'cum_n_strikes': 0,
        'cum_cg_strikes': 0,
        'ratio_n_r_cumul': 0,
        'rolling_n_ratio': 0,
        'speed': 0,
        'acceleration': 0,
        'delta_azimuth_norm': 0,
        'amp_vs_storm_mean': 1.0,
        'amp_vs_storm_max': 1.0,
        'position_in_storm': 0,
        'time_since_last_N': -1,
        'time_since_last_R': -1,
        'storm_total_lightnings': 1,
        'storm_cloud_ratio': 0.5,
        'storm_mean_amplitude': 0,
        'storm_max_amplitude': 0,
        'storm_mean_dist': 0,
        'cumulative_count': 1,
        'mean_dist_so_far': 0,
        'mean_amp_so_far': 0,
        'std_dist_so_far': 0,
    }

    for col, val in fills.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)

    valid_cols = [c for c in fills.keys() if c in df.columns]
    print(f"✅ NaN nettoyés. Vérification : {df[valid_cols].isna().sum().sum()} NaN restants dans les features.")



# =========================================================
# VÉRIFICATION DE SÉCURITÉ ANTI-LEAKAGE
# =========================================================
def check_no_future_leakage(df, group_col=STORM_GROUP_COL):
    """
    Vérifie qu'aucune feature calculée n'est 'magiquement' corrélée
    à la cible du futur de façon suspecte.
    """
    print("🔍 Vérification anti-leakage...")
    suspicious = []

    for col in df.select_dtypes(include=[np.number]).columns:
        # 🚀 CORRECTION : On ignore la colonne de groupe pour éviter le KeyError !
        if col in ['time_to_end', 'lightning_id', 'lightning_airport_id', group_col]:
            continue

        first_values = df.groupby(group_col).first()[col]
        if first_values.isna().all():
            continue

        if 'time_to_end' in df.columns:
            corr = df[col].corr(df['time_to_end'])
            # Une corrélation au-delà de 0.85 sur une métrique temporelle est suspecte
            if abs(corr) > 0.85:
                suspicious.append((col, corr))

    if suspicious:
        print("⚠️ Features suspectes (corrélation > 0.85 avec target):")
        for col, corr in sorted(suspicious, key=lambda x: -abs(x[1])):
            print(f"   {col}: corr = {corr:.3f}")
    else:
        print("✅ Aucune feature suspecte détectée. Le dataset est safe.")
format_date(df_work)
add_storm_groups_by_target(df_work, context_size=20, time_window_minutes=60)
add_time_to_end_target(df_work)
add_zone_features(df_work, ZONE)
format_alert_id(df_work)
format_last_lightning(df_work)
format_icloud(df_work)

remove_pise_2016(df_work)
remove_noise_data(df_work)
filter_useful_storms(df_work)


sort_for_sequences(df_work)
add_temporal_features(df_work)
add_kinematic_features(df_work)
add_intensity_features(df_work)
add_cumulative_features(df_work)
add_uncertainty_features(df_work)

rolling_window = 10
add_rolling_features(df_work, length=rolling_window)

add_cyclical_time_features(df_work)
add_activity_rate_features(df_work)
add_spatial_stability_features(df_work, length=rolling_window)
add_ratio_features(df_work, length=rolling_window)
add_velocity_features(df_work)
add_relative_amplitude_features(df_work)

add_position_features(df_work)
add_survival_features(df_work)
add_storm_level_features(df_work)
add_airport_code(df_work)


fill_nan_features(df_work)
check_no_future_leakage(df_work)
from sklearn.preprocessing import LabelEncoder

# 1. Créer l'encodeur
le = LabelEncoder()

# 2. Transformer la colonne 'airport' en entiers
# On crée une nouvelle colonne 'airport_id' pour garder l'originale intacte
df_work['airport_id'] = le.fit_transform(df_work['airport'])

# 3. Afficher la correspondance (pour savoir quel chiffre correspond à quel aéroport)
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Correspondance des aéroports :")
print(mapping)
# Convertir l'azimuth en radians (si ton azimuth est en degrés)
df_work['azimuth_rad'] = np.radians(df_work['azimuth'])

# Calculer X et Y (Aéroport = 0,0)
df_work['X'] = df_work['dist'] * np.sin(df_work['azimuth_rad'])
df_work['Y'] = df_work['dist'] * np.cos(df_work['azimuth_rad'])

Test sur donnée de test

In [ ]:
# 4. Passage à la Grille Temporelle (Minute par Minute)
# C'est cette étape qui génère les colonnes comme 'dist_last_5m', 'activity_count', etc.
DIST_AUDIT_KM=RISQUE_KM
df_test_minute = creer_grille_temporelle(df_work, lookback_min=5, horizon_min=30, zone_critique=ZONE, audit_dist_km=DIST_AUDIT_KM)

In [ ]:
X_test_eval = df_test_minute[features_a_entrainer]
df_test_minute['proba_brute'] = model_final.predict_proba(X_test_eval)[:, 1]
df_test_minute['proba_lissee'] = df_test_minute.groupby('storm_group_id')['proba_brute'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

In [ ]:
import numpy as np
import pandas as pd

# ==================================================
# ⚙️ CONFIGURATION DE L'ÉVALUATION FINALE
# ==================================================
SEUIL_JURY = 0.67335      # <--- Remplace par ton meilleur seuil trouvé par le scanner
MINUTES_CONFIRMATION = 10

print(f"🔍 ÉVALUATION FINALE CORRIGÉE (Audit Jury: {DIST_AUDIT_KM}km | Règle Humaine: {ZONE}km | Seuil: {SEUIL_JURY*100:.2f}%) 🔍")

# Vérification des colonnes essentielles
assert 'count_cg_audit' in df_test_minute.columns, "Il manque count_cg_audit dans df_test_minute."
assert 'count_cg_zonekm' in df_test_minute.columns, "Il manque count_cg_zonekm dans df_test_minute."

# ==========================================
# 🚨 1. LA DÉCISION DE L'IA (Verrou Irrévocable)
# ==========================================
# Décision brute et Lissage de confirmation (10 minutes)
decision_brute = (df_test_minute['proba_lissee'] > SEUIL_JURY).astype(int)
decision_ia = decision_brute.groupby(df_test_minute['storm_group_id']).transform(
    lambda x: x.rolling(window=MINUTES_CONFIRMATION, min_periods=1).max()
)

# A. Le déclencheur humain officiel (Fixe à 20 km, immuable !)
eclair_critique_humain = (df_test_minute['count_cg_zonekm'] > 0).astype(int)
alerte_declenchee = eclair_critique_humain.groupby(df_test_minute['storm_group_id']).cummax()

# B. L'IA décide d'Ouvrir (0) UNIQUEMENT si l'alerte officielle a bien sonné la minute d'avant
alerte_precedente = alerte_declenchee.groupby(df_test_minute['storm_group_id']).shift(1).fillna(0)
ordre_ouverture_ia = (alerte_precedente == 1) & (decision_ia == 0)

# C. Verrou Irrévocable : dès qu'elle ouvre pendant l'alerte, ça reste ouvert
est_ouvert_definitivement = ordre_ouverture_ia.groupby(df_test_minute['storm_group_id']).cummax()

# D. L'état final du tarmac IA : Fermé (1) SEULEMENT si alerte en cours ET pas rouvert
decision_irrevocable = ((alerte_declenchee == 1) & (~est_ouvert_definitivement)).astype(int)


# ==========================================
# 👨‍✈️ 2. LA DÉCISION DE L'HUMAIN (La Vraie Baseline)
# ==========================================
# L'humain ferme pendant EXACTEMENT 30 minutes après CHAQUE éclair <= 20 km
etat_humain_ferme = eclair_critique_humain.groupby(df_test_minute['storm_group_id']).transform(
    lambda x: x.rolling(window=30, min_periods=1).max()
)


# ==========================================
# ⏱️ 3. CALCUL DU GAIN PUR 
# ==========================================
# Gain = L'humain a peur et ferme (1), MAIS l'IA est intelligente et a ouvert (0)
minutes_gagnees = (etat_humain_ferme == 1) & (decision_irrevocable == 0) & (alerte_declenchee == 1)

# On somme les minutes par orage
gains_par_orage = minutes_gagnees.groupby(df_test_minute['storm_group_id']).sum()


# ==========================================
# ⚖️ 4. AUDIT DU RISQUE (JURY)
# ==========================================
# N_L3 : Éclairs critiques totaux pré-filtrés par la grille à DIST_AUDIT_KM
N_L3_test = df_test_minute['count_cg_audit'].sum()

# M_L3 : Éclairs ratés (L'audit du jury s'applique avec SA distance via count_cg_audit)
ouvertures_pendant_danger = (decision_irrevocable == 0) & (alerte_declenchee == 1)
M_L3_test = df_test_minute.loc[ouvertures_pendant_danger, 'count_cg_audit'].sum()

risque_test = M_L3_test / N_L3_test if N_L3_test > 0 else 0


# ==========================================
# 🏆 5. AFFICHAGE DES RÉSULTATS
# ==========================================
print("\n" + "="*80)
print(f"🏆 RÉSULTATS FINAUX (Zone Audit: {DIST_AUDIT_KM} km | Humain: 20 km)")
print("="*80)
print(f"✅ TEMPS GAGNÉ TOTAL : {gains_par_orage.sum()} minutes")
print(f"⚡ RISQUE RÉEL        : {risque_test*100:.3f}%")
print(f"   (Score Confiance   : {100 - (risque_test*100):.3f}%)")

print("-" * 80)
print(f"📊 DISTRIBUTION DU GAIN PAR ORAGE (Vérité Terrain) :")
print(f"   🔹 Moyenne       : {gains_par_orage.mean():.1f} minutes / orage")
print(f"   🔹 Médiane       : {gains_par_orage.median():.1f} minutes / orage")
print(f"   🔹 Écart-type    : {gains_par_orage.std():.1f} minutes")
print(f"   🔹 Gain Max      : {gains_par_orage.max():.1f} minutes")
print(f"   🔹 Gain Min      : {gains_par_orage.min():.1f} minutes")

print("-" * 80)
if risque_test <= RISQUE_POURCENT:
    print(f"🟢 STATUS : ACCEPTÉ PAR LE JURY (< {RISQUE_POURCENT*100} %)")
else:
    print(f"🔴 STATUS : REFUSÉ PAR LE JURY (> {RISQUE_POURCENT*100} %)")
print("="*80)

Graphique

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 GÉNÉRATION DES GRAPHIQUES DE DISTRIBUTION DES GAINS 📊")

# Configuration du style visuel (plus propre pour un rapport)
sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 6))

# ==========================================
# 📈 GRAPHIQUE 1 : L'HISTOGRAMME
# ==========================================
plt.subplot(1, 2, 1)
# On trace la distribution avec une courbe de tendance (KDE)
sns.histplot(gains_par_orage, bins=40, kde=True, color='dodgerblue', edgecolor='black')

# Ajout des lignes pour la Moyenne et la Médiane
plt.axvline(gains_par_orage.mean(), color='red', linestyle='--', linewidth=2, 
            label=f"Moyenne : {gains_par_orage.mean():.1f} min")
plt.axvline(gains_par_orage.median(), color='darkgreen', linestyle='-', linewidth=2, 
            label=f"Médiane : {gains_par_orage.median():.1f} min")

plt.title("Répartition des Gains de Temps par Orage", fontsize=14, fontweight='bold')
plt.xlabel("Temps Gagné (Minutes)", fontsize=12)
plt.ylabel("Nombre d'orages", fontsize=12)
plt.legend()

# ==========================================
# 📦 GRAPHIQUE 2 : LE BOXPLOT (Boîte à moustaches)
# ==========================================
plt.subplot(1, 2, 2)
sns.boxplot(y=gains_par_orage, color='mediumspringgreen', width=0.3)

plt.title("Dispersion et Valeurs Extrêmes (Outliers)", fontsize=14, fontweight='bold')
plt.ylabel("Temps Gagné (Minutes)", fontsize=12)

# Ajustement de l'espacement et affichage
plt.tight_layout()
plt.show()

# ==========================================
# 📝 PETIT RÉCAPITULATIF TEXTE
# ==========================================
orages_sans_gain = (gains_par_orage == 0).sum()
pct_sans_gain = (orages_sans_gain / len(gains_par_orage)) * 100

print(f"👉 Sur les {len(gains_par_orage)} orages évalués :")
print(f"   - {orages_sans_gain} orages ({pct_sans_gain:.1f}%) n'ont généré AUCUN gain (L'IA est restée aussi prudente que l'humain).")
print(f"   - Le gain maximum observé est de {gains_par_orage.max():.1f} minutes.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def tracer_analyse_orage(df, storm_id, seuil, minutes_confirmation=10):
    """
    Trace la courbe de probabilité de l'IA pour un orage spécifique,
    en coupant tout ce qui se passe avant le premier éclair d'alerte.
    Version épurée : sans légende et sans caractères spéciaux.
    """
    # 1. Filtrer les données pour l'orage demandé
    df_orage = df[df['storm_group_id'] == storm_id].copy()
    
    if df_orage.empty:
        print(f"Attention : L'orage avec l'ID {storm_id} n'a pas été trouvé dans le dataset.")
        return
        
    # ==========================================
    # COUPER L'AVANT-TEMPÊTE (Focus sur l'Alerte)
    # ==========================================
    if 'count_cg_zonekm' in df_orage.columns and df_orage['count_cg_zonekm'].sum() > 0:
        # Trouver l'index de la ligne du tout premier éclair critique (Le déclencheur)
        idx_premier_eclair = df_orage[df_orage['count_cg_zonekm'] > 0].index[0]
        
        # On ne garde que les données à partir de cette minute, et on remet l'index à zéro
        df_orage = df_orage.loc[idx_premier_eclair:].reset_index(drop=True)
    else:
        print("Attention : Aucun éclair à < zonekm pour cet orage. Affichage complet.")

    # L'axe X (minutes) démarre maintenant exactement au moment de l'alerte (T = 0)
    minutes_ecoulees = np.arange(len(df_orage))
    
    # ==========================================
    # RECALCUL DES DÉCISIONS POUR LE GRAPHIQUE
    # ==========================================
    decision_brute = (df_orage['proba_lissee'] > seuil).astype(int)
    decision_ia = decision_brute.rolling(window=minutes_confirmation, min_periods=1).max()
    
    mask_danger = (decision_brute == 1)
    mask_verrou = (decision_brute == 0) & (decision_ia == 1)
    mask_ouvert = (decision_ia == 0)
    
    # 2. Configuration de la figure
    sns.set_theme(style="whitegrid")
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # ==========================================
    # AXE 1 : PROBABILITÉS DE L'IA ET ZONES
    # ==========================================
    ax1.plot(minutes_ecoulees, df_orage['proba_lissee'], color='dodgerblue', linewidth=2.5)
    
    ax1.axhline(y=seuil, color='red', linestyle='--', linewidth=2)
    
    ax1.fill_between(minutes_ecoulees, df_orage['proba_lissee'], seuil, 
                     where=mask_danger, color='red', alpha=0.15)
                     
    ax1.fill_between(minutes_ecoulees, df_orage['proba_lissee'], seuil, 
                     where=mask_verrou, color='gold', alpha=0.3)

    ax1.fill_between(minutes_ecoulees, df_orage['proba_lissee'], seuil, 
                     where=mask_ouvert, color='mediumspringgreen', alpha=0.15)

    # ==========================================
    # FIN RÉELLE DE L'ORAGE ET HUMAIN
    # ==========================================
    limite_max_X = len(df_orage)

    if 'count_cg_zonekm' in df_orage.columns and df_orage['count_cg_zonekm'].sum() > 0:
        indices_eclairs = np.where(df_orage['count_cg_zonekm'] > 0)[0]
        derniere_minute_eclair = indices_eclairs[-1]
        
        ax1.axvline(x=derniere_minute_eclair, color='black', linestyle='-.', linewidth=2.5)
        
        minute_humain = derniere_minute_eclair + 30
        
        ax1.axvline(x=minute_humain, color='purple', linestyle=':', linewidth=2.5)
        
        # On s'assure que le graphique aille au moins jusqu'à l'ouverture humaine
        limite_max_X = max(limite_max_X, minute_humain + 5)

    ax1.set_xlabel("Temps ecoule depuis la 1ere alerte (Minutes)", fontsize=12)
    ax1.set_ylabel("Probabilite de Danger", fontsize=12, color='dodgerblue')
    ax1.set_ylim(0, 1.05)
    ax1.set_xlim(0, limite_max_X)
    ax1.tick_params(axis='y', labelcolor='dodgerblue')
    
    # ==========================================
    # AXE 2 : LES ÉCLAIRS RÉELS (Barres de fond)
    # ==========================================
    if 'count_cg_zonekm' in df_orage.columns:
        ax2 = ax1.twinx()
        
        ax2.bar(minutes_ecoulees, df_orage['count_cg_zonekm'], color='gray', alpha=0.4, width=1.0)
        
        if 'count_cg_3km' in df_orage.columns:
            ax2.bar(minutes_ecoulees, df_orage['count_cg_3km'], color='crimson', alpha=0.9, width=1.0)
        
        max_eclairs = df_orage['count_cg_zonekm'].max()
        ax2.set_ylim(0, max_eclairs * 4 if max_eclairs > 0 else 10) 
        
        ax2.set_ylabel("Nombre d'eclairs par minute", fontsize=12, color='dimgray')
        ax2.tick_params(axis='y', labelcolor='dimgray')

    # ==========================================
    # FINITIONS
    # ==========================================
    # Titre épuré sans caractères spéciaux
    plt.title(f"Radiographie de l'Orage #{storm_id} (Seuil IA: {seuil:.3f})", 
              fontsize=16, fontweight='bold', y=1.05)
    
    plt.tight_layout()
    plt.show()
tracer_analyse_orage(df_test_minute, storm_id=710, seuil=SEUIL_JURY, minutes_confirmation=10)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

def generer_legende_seule(seuil=0.342, minutes_confirmation=10):
    """
    Génère une image contenant uniquement la légende explicative de l'analyse d'orage.
    Parfait pour exporter vers une présentation ou un rapport.
    """
    # 1. Création des "Handles" (Les symboles visuels de la légende)
    elements_legende = [
        # Courbes et Lignes
        Line2D([0], [0], color='dodgerblue', lw=2.5, label="Probabilite IA (Danger)"),
        Line2D([0], [0], color='red', lw=2, linestyle='--', label=f"Seuil d'Ouverture ({seuil*100:.1f}%)"),
        Line2D([0], [0], color='black', lw=2.5, linestyle='-.', label="Fin reelle de l'orage (Dernier eclair)"),
        Line2D([0], [0], color='purple', lw=2.5, linestyle=':', label="Ouverture Humaine (Règle des 30 min)"),
        
        # Zones de couleur (Fill_between)
        Patch(facecolor='red', alpha=0.15, edgecolor='red', label="Zone de Danger (> Seuil)"),
        Patch(facecolor='gold', alpha=0.3, edgecolor='orange', label=f"Verrou de securite ({minutes_confirmation} min)"),
        Patch(facecolor='mediumspringgreen', alpha=0.15, edgecolor='green', label="Ouverture Confirmee IA (Safe)"),
        
        # Barres d'éclairs
        Patch(facecolor='gray', alpha=0.4, label="Impacts de foudre (< 20km)"),
        Patch(facecolor='crimson', alpha=0.9, label="Impacts CRITIQUES (< 3km)")
    ]

    # 2. Configuration d'une figure "vide" juste pour la légende
    fig, ax = plt.subplots(figsize=(8, 4))
    
    # On cache complètement les axes et le fond
    ax.axis('off') 

    # 3. Création de la légende au centre de cette figure vide
    legende = ax.legend(handles=elements_legende, 
                        loc='center', 
                        fontsize=12, 
                        frameon=True, 
                        shadow=True, 
                        borderpad=1.5, 
                        labelspacing=1.2,
                        ncol=2) # ncol=2 met la légende sur 2 colonnes pour faire plus propre
    
    # Rendre le titre de la légende un peu stylé
    legende.set_title("Légende de la Radiographie", prop={'size': 14, 'weight': 'bold'})

    plt.tight_layout()
    plt.show()

# Appeler la fonction pour afficher juste la légende
generer_legende_seule(seuil=SEUIL_JURY, minutes_confirmation=10)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 GÉNÉRATION DES HISTOGRAMMES SÉPARÉS PAR AÉROPORT 📊")

# 1. On récupère le nom de l'aéroport pour chaque orage
mapping_airport = df_test_minute.groupby('storm_group_id')['airport_id'].first()

# On crée un DataFrame de résultats
df_results = pd.DataFrame({
    'Gain_Brut': gains_par_orage
})

# On fusionne avec les aéroports
df_results = df_results.join(mapping_airport)

# Traduction des IDs en vrais noms d'aéroports
dictionnaire_aeroports = {0: 'Ajaccio', 1: 'Bastia', 2: 'Biarritz', 3: 'Nantes', 4: 'Pise'}
df_results['airport_name'] = df_results['airport_id'].map(dictionnaire_aeroports)

# 2. APPLICATION DE LA CORRECTION (-1)
df_results['Gain_Corrige'] = (df_results['Gain_Brut'] - 1).clip(lower=0)

# ==========================================
# 📈 GRAPHIQUE : GRILLE D'HISTOGRAMMES (FacetGrid)
# ==========================================
sns.set_theme(style="whitegrid")

# sns.displot gère sa propre figure, on lui dit de séparer par colonnes (col)
g = sns.displot(
    data=df_results, 
    x='Gain_Corrige', 
    col='airport_name',    
    col_wrap=3,            
    hue='airport_name',    
    bins=60,               # 👈 LA MAGIE EST ICI : On passe de 30 à 60 barres !
    kde=True,              
    edgecolor='black',
    linewidth=0.5,         # On garde un contour fin pour que ça reste lisible avec beaucoup de barres
    height=4,              
    aspect=1.2,            
    legend=False           
)

# 🎨 Personnalisation des titres de chaque petit graphique
g.set_titles("{col_name}", size=14, weight='bold')

# Noms des axes
g.set_axis_labels("Temps Gagné (Minutes)", "Nombre d'orages", fontsize=12)

# On force tous les axes X à démarrer à 0
g.set(xlim=(0, None))

# Titre global tout en haut
plt.suptitle("Distribution des Gains Corrigés par Aéroport", fontsize=18, fontweight='bold', y=1.05)

plt.show()

# ==========================================
# 📝 STATISTIQUES EXACTES (Moyennes & Médianes)
# ==========================================
print("\n🏆 STATISTIQUES EXACTES PAR AÉROPORT (Gains Corrigés) 🏆")
print("-" * 65)

stats_aeroport = df_results.groupby('airport_name')['Gain_Corrige'].agg(['count', 'mean', 'median', 'max']).round(1)
stats_aeroport.columns = ['Nb Orages', 'Moyenne (min)', 'Médiane (min)', 'Gain Max (min)']
stats_aeroport = stats_aeroport.sort_values(by='Moyenne (min)', ascending=False)
stats_aeroport.index.name = None

print(stats_aeroport.to_string())
print("-" * 65)

moyenne_globale = df_results['Gain_Corrige'].mean()
mediane_globale = df_results['Gain_Corrige'].median()
print(f"🌍 GLOBAL -> Moyenne : {moyenne_globale:.1f} min | Médiane : {mediane_globale:.1f} min")